In [2]:
## no lookback with 1y, 5y and full timeframe comparison

## Statistical level hit analysis - COMPLETE TIMEFRAME COMPARISON (ALL STATS)
## 
## This script calculates ALL statistics from historical data comparing:
## - Last 1 year
## - Last 5 years  
## - Full dataset (12 years)
##
## For ALL timeframes: Hourly (24 slots), 4H (6 slots), 6H (4 slots), 8H (3 slots), 12H (2 slots), Daily
##
## Trading Day: 18:00 (6 PM) to 17:00 (5 PM) next day

import pandas as pd
import numpy as np
from collections import defaultdict
import sys
from datetime import datetime, timedelta

print("=" * 80)
print("STATISTICAL LEVEL ANALYSIS - COMPLETE TIMEFRAME COMPARISON (ALL STATS)")
print("=" * 80)
sys.stdout.flush()

# Load the data
file_path = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
print(f"\nLoading data from: {file_path}")
sys.stdout.flush()
df = pd.read_csv(file_path)
print(f"✓ Loaded {len(df):,} rows")
sys.stdout.flush()

# Combine Date and Time into a datetime column
print("\nCreating datetime index...")
sys.stdout.flush()
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)
print(f"✓ Datetime index created")
sys.stdout.flush()

# Filter data from 2013 to 2025
print("\nFiltering to 2013-2025...")
sys.stdout.flush()
start_date = pd.to_datetime('2013-01-01')
end_date = pd.to_datetime('2025-12-31')
df = df.loc[start_date:end_date]
print(f"✓ Filtered to {len(df):,} rows")
sys.stdout.flush()

# Resample to 5-minute bars for performance
print("\nResampling to 5-minute bars...")
sys.stdout.flush()
df = df.resample('5min').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last'
}).dropna()
print(f"✓ Resampled to {len(df):,} 5-minute bars")
sys.stdout.flush()

# Define timeframe cutoffs
current_date = df.index.max()
one_year_ago = current_date - pd.Timedelta(days=365)
five_years_ago = current_date - pd.Timedelta(days=365*5)

print(f"\nTimeframe cutoffs:")
print(f"  Full dataset: {df.index.min().date()} to {df.index.max().date()}")
print(f"  5-year: {five_years_ago.date()} to {df.index.max().date()}")
print(f"  1-year: {one_year_ago.date()} to {df.index.max().date()}")
sys.stdout.flush()

# Create filtered datasets
df_1y = df[df.index >= one_year_ago].copy()
df_5y = df[df.index >= five_years_ago].copy()
df_full = df.copy()

print(f"\nDataset sizes:")
print(f"  Full: {len(df_full):,} bars")
print(f"  5-year: {len(df_5y):,} bars")
print(f"  1-year: {len(df_1y):,} bars")
sys.stdout.flush()

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_trading_hour(timestamp):
    """Get the trading hour (0-23) where 18:00 is hour 0"""
    hour = timestamp.hour
    if hour >= 18:
        return hour - 18
    else:
        return hour + 6

def get_4h_slot(timestamp):
    """Get the 4H slot (0-5)"""
    trading_hour = get_trading_hour(timestamp)
    return trading_hour // 4

def get_6h_slot(timestamp):
    """Get the 6H slot (0-3)"""
    trading_hour = get_trading_hour(timestamp)
    return trading_hour // 6

def get_8h_slot(timestamp):
    """Get the 8H slot (0-2)"""
    trading_hour = get_trading_hour(timestamp)
    return trading_hour // 8

def get_12h_slot(timestamp):
    """Get the 12H slot (0-1)"""
    trading_hour = get_trading_hour(timestamp)
    return trading_hour // 12

def calculate_manipulation_wick(open_price, high, low, close):
    """Manipulation wick: the wick against the candle direction"""
    if close > open_price:
        return open_price - low
    else:
        return high - open_price

def calculate_distribution_distance(open_price, high, low, close):
    """Distribution: extension in the candle's direction"""
    if close > open_price:
        return high - open_price
    else:
        return open_price - low

def calculate_max_range_from_open(open_price, high, low):
    """Maximum distance from open in either direction"""
    return max(abs(high - open_price), abs(open_price - low))

def calculate_high_range_from_open(open_price, high):
    """Distance from open to high"""
    return high - open_price

def calculate_low_range_from_open(open_price, low):
    """Distance from open to low"""
    return open_price - low

def calculate_percentage_from_open(distance, open_price):
    """Calculate percentage distance from open"""
    if open_price == 0:
        return 0
    return (distance / open_price) * 100

def create_stats_dict():
    """Create a statistics dictionary with ALL metrics"""
    return {
        'total_candles': 0,
        
        # Level hits - MEDIAN based
        'manipulation_high_hit_median': 0,
        'manipulation_low_hit_median': 0,
        'distribution_high_hit_median': 0,
        'distribution_low_hit_median': 0,
        'max_range_high_hit_median': 0,
        'max_range_low_hit_median': 0,
        
        # Level hits - MEAN based
        'manipulation_high_hit_mean': 0,
        'manipulation_low_hit_mean': 0,
        'distribution_high_hit_mean': 0,
        'distribution_low_hit_mean': 0,
        'max_range_high_hit_mean': 0,
        'max_range_low_hit_mean': 0,
        
        # Percentile hits
        'percentile_75_high_hit': 0,
        'percentile_25_high_hit': 0,
        'percentile_75_low_hit': 0,
        'percentile_25_low_hit': 0,
        
        # Close position relative to levels - MEDIAN based
        'close_above_manipulation_high_median': 0,
        'close_below_manipulation_low_median': 0,
        'close_above_distribution_high_median': 0,
        'close_below_distribution_low_median': 0,
        'close_above_max_range_high_median': 0,
        'close_below_max_range_low_median': 0,
        
        # Close position relative to levels - MEAN based
        'close_above_manipulation_high_mean': 0,
        'close_below_manipulation_low_mean': 0,
        'close_above_distribution_high_mean': 0,
        'close_below_distribution_low_mean': 0,
        'close_above_max_range_high_mean': 0,
        'close_below_max_range_low_mean': 0,
        
        # Close position relative to percentiles
        'close_above_percentile_75_high': 0,
        'close_below_percentile_75_low': 0,
        
        # Directional bias
        'bullish_candles': 0,
        'bearish_candles': 0,
        'doji_candles': 0,
        
        # Strong moves
        'close_top_25_percent': 0,
        'close_bottom_25_percent': 0,
        
        # Level sequencing - MEDIAN based
        'manip_high_then_dist_high_median': 0,
        'manip_high_then_reverse_median': 0,
        'manip_low_then_dist_low_median': 0,
        'manip_low_then_reverse_median': 0,
        
        # Level sequencing - MEAN based
        'manip_high_then_dist_high_mean': 0,
        'manip_high_then_reverse_mean': 0,
        'manip_low_then_dist_low_mean': 0,
        'manip_low_then_reverse_mean': 0,
        
        # Double-sided hits - MEDIAN based
        'both_manip_levels_hit_median': 0,
        'both_dist_levels_hit_median': 0,
        
        # Double-sided hits - MEAN based
        'both_manip_levels_hit_mean': 0,
        'both_dist_levels_hit_mean': 0,
        
        # Conditional probabilities - MEDIAN based
        'bullish_and_hit_dist_high_median': 0,
        'bearish_and_hit_dist_low_median': 0,
        'manip_low_hit_and_closed_bullish_median': 0,
        'manip_high_hit_and_closed_bearish_median': 0,
        
        # Conditional probabilities - MEAN based
        'bullish_and_hit_dist_high_mean': 0,
        'bearish_and_hit_dist_low_mean': 0,
        'manip_low_hit_and_closed_bullish_mean': 0,
        'manip_high_hit_and_closed_bearish_mean': 0,
        
        # Percentage tracking
        'manip_wick_percentages': [],
        'dist_distance_percentages': [],
        'max_range_percentages': [],
        'high_range_percentages': [],
        'low_range_percentages': [],
    }

# ============================================================
# ANALYSIS FUNCTION
# ============================================================

def analyze_timeframe(data, timeframe_label):
    """Analyze a single timeframe - ALL candle types with ALL statistics"""
    print(f"\n{'=' * 80}")
    print(f"ANALYZING: {timeframe_label}")
    print(f"{'=' * 80}")
    sys.stdout.flush()
    
    # Add columns
    data['trading_hour'] = data.index.map(get_trading_hour)
    data['trading_4h_slot'] = data.index.map(get_4h_slot)
    data['trading_6h_slot'] = data.index.map(get_6h_slot)
    data['trading_8h_slot'] = data.index.map(get_8h_slot)
    data['trading_12h_slot'] = data.index.map(get_12h_slot)
    data['NY_Trading_Day'] = (data.index - pd.Timedelta(hours=6)).normalize()
    
    data['hour_id'] = data['NY_Trading_Day'].astype(str) + '_' + data['trading_hour'].astype(str)
    data['4h_id'] = data['NY_Trading_Day'].astype(str) + '_' + data['trading_4h_slot'].astype(str)
    data['6h_id'] = data['NY_Trading_Day'].astype(str) + '_' + data['trading_6h_slot'].astype(str)
    data['8h_id'] = data['NY_Trading_Day'].astype(str) + '_' + data['trading_8h_slot'].astype(str)
    data['12h_id'] = data['NY_Trading_Day'].astype(str) + '_' + data['trading_12h_slot'].astype(str)
    data['daily_id'] = data['NY_Trading_Day'].astype(str)
    
    # Initialize storage
    hourly_stats = {hour: create_stats_dict() for hour in range(24)}
    four_hour_stats = {slot: create_stats_dict() for slot in range(6)}
    six_hour_stats = {slot: create_stats_dict() for slot in range(4)}
    eight_hour_stats = {slot: create_stats_dict() for slot in range(3)}
    twelve_hour_stats = {slot: create_stats_dict() for slot in range(2)}
    daily_stats = create_stats_dict()
    
    def process_candles(groups, stats_dict, slot_col=None, num_slots=None, label=""):
        """Generic function to process candles with ALL statistics"""
        print(f"  Processing {label}...")
        sys.stdout.flush()
        
        if num_slots is None:
            # Daily - no slots
            candles = groups.copy()
            
            # Calculate metrics
            candles['manip_wick'] = candles.apply(
                lambda row: calculate_manipulation_wick(row['Open'], row['High'], row['Low'], row['Close']), 
                axis=1
            )
            candles['dist_distance'] = candles.apply(
                lambda row: calculate_distribution_distance(row['Open'], row['High'], row['Low'], row['Close']), 
                axis=1
            )
            candles['max_range'] = candles.apply(
                lambda row: calculate_max_range_from_open(row['Open'], row['High'], row['Low']), 
                axis=1
            )
            candles['high_range'] = candles.apply(
                lambda row: calculate_high_range_from_open(row['Open'], row['High']), 
                axis=1
            )
            candles['low_range'] = candles.apply(
                lambda row: calculate_low_range_from_open(row['Open'], row['Low']), 
                axis=1
            )
            candles['candle_range'] = candles['High'] - candles['Low']
            
            # Calculate percentages
            candles['manip_wick_pct'] = candles.apply(
                lambda row: calculate_percentage_from_open(row['manip_wick'], row['Open']),
                axis=1
            )
            candles['dist_distance_pct'] = candles.apply(
                lambda row: calculate_percentage_from_open(row['dist_distance'], row['Open']),
                axis=1
            )
            candles['max_range_pct'] = candles.apply(
                lambda row: calculate_percentage_from_open(row['max_range'], row['Open']),
                axis=1
            )
            candles['high_range_pct'] = candles.apply(
                lambda row: calculate_percentage_from_open(row['high_range'], row['Open']),
                axis=1
            )
            candles['low_range_pct'] = candles.apply(
                lambda row: calculate_percentage_from_open(row['low_range'], row['Open']),
                axis=1
            )
            
            # Store percentages
            stats_dict['manip_wick_percentages'] = candles['manip_wick_pct'].tolist()
            stats_dict['dist_distance_percentages'] = candles['dist_distance_pct'].tolist()
            stats_dict['max_range_percentages'] = candles['max_range_pct'].tolist()
            stats_dict['high_range_percentages'] = candles['high_range_pct'].tolist()
            stats_dict['low_range_percentages'] = candles['low_range_pct'].tolist()
            
            # Calculate global statistics
            all_manip = np.array(candles['manip_wick'])
            all_dist = np.array(candles['dist_distance'])
            all_max = np.array(candles['max_range'])
            all_high_ranges = np.array(candles['high_range'])
            all_low_ranges = np.array(candles['low_range'])
            
            median_manip = np.median(all_manip)
            mean_manip = np.mean(all_manip)
            median_dist = np.median(all_dist)
            mean_dist = np.mean(all_dist)
            median_max = np.median(all_max)
            mean_max = np.mean(all_max)
            
            # Percentiles
            percentile_75_high = np.percentile(all_high_ranges, 75)
            percentile_25_high = np.percentile(all_high_ranges, 25)
            percentile_75_low = np.percentile(all_low_ranges, 75)
            percentile_25_low = np.percentile(all_low_ranges, 25)
            
            # Analyze each candle
            total = len(candles)
            stats_dict['total_candles'] = total
            
            for idx, row in candles.iterrows():
                current_open = row['Open']
                current_high = row['High']
                current_low = row['Low']
                current_close = row['Close']
                current_range = row['candle_range']
                
                # Project levels from MEDIAN statistics
                manipulation_high_med = current_open + median_manip
                manipulation_low_med = current_open - median_manip
                distribution_high_med = current_open + median_dist + median_manip
                distribution_low_med = current_open - median_dist - median_manip
                max_range_high_med = current_open + median_max
                max_range_low_med = current_open - median_max
                
                # Project levels from MEAN statistics
                manipulation_high_mean = current_open + mean_manip
                manipulation_low_mean = current_open - mean_manip
                distribution_high_mean = current_open + mean_dist + mean_manip
                distribution_low_mean = current_open - mean_dist - mean_manip
                max_range_high_mean = current_open + mean_max
                max_range_low_mean = current_open - mean_max
                
                # Percentile levels (same for both)
                percentile_75_high_level = current_open + percentile_75_high
                percentile_25_high_level = current_open + percentile_25_high
                percentile_75_low_level = current_open - percentile_75_low
                percentile_25_low_level = current_open - percentile_25_low
                
                # Directional bias
                is_bullish_candle = current_close > current_open
                if is_bullish_candle:
                    stats_dict['bullish_candles'] += 1
                else:
                    stats_dict['bearish_candles'] += 1
                
                # Check for doji
                if current_range > 0:
                    body_pct = abs(current_close - current_open) / current_range
                    if body_pct < 0.1:
                        stats_dict['doji_candles'] += 1
                
                # Strong moves
                if current_range > 0:
                    close_pos = (current_close - current_low) / current_range
                    if close_pos >= 0.75:
                        stats_dict['close_top_25_percent'] += 1
                    elif close_pos <= 0.25:
                        stats_dict['close_bottom_25_percent'] += 1
                
                # Check if MEDIAN-based levels were hit
                manip_high_hit_med = current_high >= manipulation_high_med
                manip_low_hit_med = current_low <= manipulation_low_med
                dist_high_hit_med = current_high >= distribution_high_med
                dist_low_hit_med = current_low <= distribution_low_med
                
                if manip_high_hit_med:
                    stats_dict['manipulation_high_hit_median'] += 1
                if manip_low_hit_med:
                    stats_dict['manipulation_low_hit_median'] += 1
                if dist_high_hit_med:
                    stats_dict['distribution_high_hit_median'] += 1
                if dist_low_hit_med:
                    stats_dict['distribution_low_hit_median'] += 1
                if current_high >= max_range_high_med:
                    stats_dict['max_range_high_hit_median'] += 1
                if current_low <= max_range_low_med:
                    stats_dict['max_range_low_hit_median'] += 1
                
                # Check if MEAN-based levels were hit
                manip_high_hit_mean = current_high >= manipulation_high_mean
                manip_low_hit_mean = current_low <= manipulation_low_mean
                dist_high_hit_mean = current_high >= distribution_high_mean
                dist_low_hit_mean = current_low <= distribution_low_mean
                
                if manip_high_hit_mean:
                    stats_dict['manipulation_high_hit_mean'] += 1
                if manip_low_hit_mean:
                    stats_dict['manipulation_low_hit_mean'] += 1
                if dist_high_hit_mean:
                    stats_dict['distribution_high_hit_mean'] += 1
                if dist_low_hit_mean:
                    stats_dict['distribution_low_hit_mean'] += 1
                if current_high >= max_range_high_mean:
                    stats_dict['max_range_high_hit_mean'] += 1
                if current_low <= max_range_low_mean:
                    stats_dict['max_range_low_hit_mean'] += 1
                
                # Percentile hits (same for both)
                if current_high >= percentile_75_high_level:
                    stats_dict['percentile_75_high_hit'] += 1
                if current_high >= percentile_25_high_level:
                    stats_dict['percentile_25_high_hit'] += 1
                if current_low <= percentile_75_low_level:
                    stats_dict['percentile_75_low_hit'] += 1
                if current_low <= percentile_25_low_level:
                    stats_dict['percentile_25_low_hit'] += 1
                
                # Level sequencing - MEDIAN
                if manip_high_hit_med:
                    if dist_high_hit_med:
                        stats_dict['manip_high_then_dist_high_median'] += 1
                    if manip_low_hit_med:
                        stats_dict['manip_high_then_reverse_median'] += 1
                
                if manip_low_hit_med:
                    if dist_low_hit_med:
                        stats_dict['manip_low_then_dist_low_median'] += 1
                    if manip_high_hit_med:
                        stats_dict['manip_low_then_reverse_median'] += 1
                
                # Level sequencing - MEAN
                if manip_high_hit_mean:
                    if dist_high_hit_mean:
                        stats_dict['manip_high_then_dist_high_mean'] += 1
                    if manip_low_hit_mean:
                        stats_dict['manip_high_then_reverse_mean'] += 1
                
                if manip_low_hit_mean:
                    if dist_low_hit_mean:
                        stats_dict['manip_low_then_dist_low_mean'] += 1
                    if manip_high_hit_mean:
                        stats_dict['manip_low_then_reverse_mean'] += 1
                
                # Double-sided hits - MEDIAN
                if manip_high_hit_med and manip_low_hit_med:
                    stats_dict['both_manip_levels_hit_median'] += 1
                if dist_high_hit_med and dist_low_hit_med:
                    stats_dict['both_dist_levels_hit_median'] += 1
                
                # Double-sided hits - MEAN
                if manip_high_hit_mean and manip_low_hit_mean:
                    stats_dict['both_manip_levels_hit_mean'] += 1
                if dist_high_hit_mean and dist_low_hit_mean:
                    stats_dict['both_dist_levels_hit_mean'] += 1
                
                # Conditional probabilities - MEDIAN
                if is_bullish_candle and dist_high_hit_med:
                    stats_dict['bullish_and_hit_dist_high_median'] += 1
                if not is_bullish_candle and dist_low_hit_med:
                    stats_dict['bearish_and_hit_dist_low_median'] += 1
                if manip_low_hit_med and is_bullish_candle:
                    stats_dict['manip_low_hit_and_closed_bullish_median'] += 1
                if manip_high_hit_med and not is_bullish_candle:
                    stats_dict['manip_high_hit_and_closed_bearish_median'] += 1
                
                # Conditional probabilities - MEAN
                if is_bullish_candle and dist_high_hit_mean:
                    stats_dict['bullish_and_hit_dist_high_mean'] += 1
                if not is_bullish_candle and dist_low_hit_mean:
                    stats_dict['bearish_and_hit_dist_low_mean'] += 1
                if manip_low_hit_mean and is_bullish_candle:
                    stats_dict['manip_low_hit_and_closed_bullish_mean'] += 1
                if manip_high_hit_mean and not is_bullish_candle:
                    stats_dict['manip_high_hit_and_closed_bearish_mean'] += 1
                
                # Check where close is relative to MEDIAN levels
                if current_close > manipulation_high_med:
                    stats_dict['close_above_manipulation_high_median'] += 1
                if current_close < manipulation_low_med:
                    stats_dict['close_below_manipulation_low_median'] += 1
                if current_close > distribution_high_med:
                    stats_dict['close_above_distribution_high_median'] += 1
                if current_close < distribution_low_med:
                    stats_dict['close_below_distribution_low_median'] += 1
                if current_close > max_range_high_med:
                    stats_dict['close_above_max_range_high_median'] += 1
                if current_close < max_range_low_med:
                    stats_dict['close_below_max_range_low_median'] += 1
                
                # Check where close is relative to MEAN levels
                if current_close > manipulation_high_mean:
                    stats_dict['close_above_manipulation_high_mean'] += 1
                if current_close < manipulation_low_mean:
                    stats_dict['close_below_manipulation_low_mean'] += 1
                if current_close > distribution_high_mean:
                    stats_dict['close_above_distribution_high_mean'] += 1
                if current_close < distribution_low_mean:
                    stats_dict['close_below_distribution_low_mean'] += 1
                if current_close > max_range_high_mean:
                    stats_dict['close_above_max_range_high_mean'] += 1
                if current_close < max_range_low_mean:
                    stats_dict['close_below_max_range_low_mean'] += 1
                
                # Check where close is relative to percentiles
                if current_close > percentile_75_high_level:
                    stats_dict['close_above_percentile_75_high'] += 1
                if current_close < percentile_75_low_level:
                    stats_dict['close_below_percentile_75_low'] += 1
                    
        else:
            # Multi-slot (hourly, 4H, 6H, 8H, 12H)
            for slot in range(num_slots):
                slot_candles = groups[groups[slot_col] == slot].copy()
                
                if len(slot_candles) == 0:
                    continue
                
                # Calculate metrics
                slot_candles['manip_wick'] = slot_candles.apply(
                    lambda row: calculate_manipulation_wick(row['Open'], row['High'], row['Low'], row['Close']), 
                    axis=1
                )
                slot_candles['dist_distance'] = slot_candles.apply(
                    lambda row: calculate_distribution_distance(row['Open'], row['High'], row['Low'], row['Close']), 
                    axis=1
                )
                slot_candles['max_range'] = slot_candles.apply(
                    lambda row: calculate_max_range_from_open(row['Open'], row['High'], row['Low']), 
                    axis=1
                )
                slot_candles['high_range'] = slot_candles.apply(
                    lambda row: calculate_high_range_from_open(row['Open'], row['High']), 
                    axis=1
                )
                slot_candles['low_range'] = slot_candles.apply(
                    lambda row: calculate_low_range_from_open(row['Open'], row['Low']), 
                    axis=1
                )
                slot_candles['candle_range'] = slot_candles['High'] - slot_candles['Low']
                
                # Calculate percentages
                slot_candles['manip_wick_pct'] = slot_candles.apply(
                    lambda row: calculate_percentage_from_open(row['manip_wick'], row['Open']),
                    axis=1
                )
                slot_candles['dist_distance_pct'] = slot_candles.apply(
                    lambda row: calculate_percentage_from_open(row['dist_distance'], row['Open']),
                    axis=1
                )
                slot_candles['max_range_pct'] = slot_candles.apply(
                    lambda row: calculate_percentage_from_open(row['max_range'], row['Open']),
                    axis=1
                )
                slot_candles['high_range_pct'] = slot_candles.apply(
                    lambda row: calculate_percentage_from_open(row['high_range'], row['Open']),
                    axis=1
                )
                slot_candles['low_range_pct'] = slot_candles.apply(
                    lambda row: calculate_percentage_from_open(row['low_range'], row['Open']),
                    axis=1
                )
                
                # Store percentages
                stats_dict[slot]['manip_wick_percentages'] = slot_candles['manip_wick_pct'].tolist()
                stats_dict[slot]['dist_distance_percentages'] = slot_candles['dist_distance_pct'].tolist()
                stats_dict[slot]['max_range_percentages'] = slot_candles['max_range_pct'].tolist()
                stats_dict[slot]['high_range_percentages'] = slot_candles['high_range_pct'].tolist()
                stats_dict[slot]['low_range_percentages'] = slot_candles['low_range_pct'].tolist()
                
                # Calculate global statistics for this slot
                all_manip = np.array(slot_candles['manip_wick'])
                all_dist = np.array(slot_candles['dist_distance'])
                all_max = np.array(slot_candles['max_range'])
                all_high_ranges = np.array(slot_candles['high_range'])
                all_low_ranges = np.array(slot_candles['low_range'])
                
                median_manip = np.median(all_manip)
                mean_manip = np.mean(all_manip)
                median_dist = np.median(all_dist)
                mean_dist = np.mean(all_dist)
                median_max = np.median(all_max)
                mean_max = np.mean(all_max)
                
                # Percentiles
                percentile_75_high = np.percentile(all_high_ranges, 75)
                percentile_25_high = np.percentile(all_high_ranges, 25)
                percentile_75_low = np.percentile(all_low_ranges, 75)
                percentile_25_low = np.percentile(all_low_ranges, 25)
                
                # Analyze each candle
                total = len(slot_candles)
                stats_dict[slot]['total_candles'] = total
                
                for idx, row in slot_candles.iterrows():
                    current_open = row['Open']
                    current_high = row['High']
                    current_low = row['Low']
                    current_close = row['Close']
                    current_range = row['candle_range']
                    
                    # Project levels from MEDIAN statistics
                    manipulation_high_med = current_open + median_manip
                    manipulation_low_med = current_open - median_manip
                    distribution_high_med = current_open + median_dist + median_manip
                    distribution_low_med = current_open - median_dist - median_manip
                    max_range_high_med = current_open + median_max
                    max_range_low_med = current_open - median_max
                    
                    # Project levels from MEAN statistics
                    manipulation_high_mean = current_open + mean_manip
                    manipulation_low_mean = current_open - mean_manip
                    distribution_high_mean = current_open + mean_dist + mean_manip
                    distribution_low_mean = current_open - mean_dist - mean_manip
                    max_range_high_mean = current_open + mean_max
                    max_range_low_mean = current_open - mean_max
                    
                    # Percentile levels (same for both)
                    percentile_75_high_level = current_open + percentile_75_high
                    percentile_25_high_level = current_open + percentile_25_high
                    percentile_75_low_level = current_open - percentile_75_low
                    percentile_25_low_level = current_open - percentile_25_low
                    
                    # Directional bias
                    is_bullish_candle = current_close > current_open
                    if is_bullish_candle:
                        stats_dict[slot]['bullish_candles'] += 1
                    else:
                        stats_dict[slot]['bearish_candles'] += 1
                    
                    # Check for doji
                    if current_range > 0:
                        body_pct = abs(current_close - current_open) / current_range
                        if body_pct < 0.1:
                            stats_dict[slot]['doji_candles'] += 1
                    
                    # Strong moves
                    if current_range > 0:
                        close_pos = (current_close - current_low) / current_range
                        if close_pos >= 0.75:
                            stats_dict[slot]['close_top_25_percent'] += 1
                        elif close_pos <= 0.25:
                            stats_dict[slot]['close_bottom_25_percent'] += 1
                    
                    # Check if MEDIAN-based levels were hit
                    manip_high_hit_med = current_high >= manipulation_high_med
                    manip_low_hit_med = current_low <= manipulation_low_med
                    dist_high_hit_med = current_high >= distribution_high_med
                    dist_low_hit_med = current_low <= distribution_low_med
                    
                    if manip_high_hit_med:
                        stats_dict[slot]['manipulation_high_hit_median'] += 1
                    if manip_low_hit_med:
                        stats_dict[slot]['manipulation_low_hit_median'] += 1
                    if dist_high_hit_med:
                        stats_dict[slot]['distribution_high_hit_median'] += 1
                    if dist_low_hit_med:
                        stats_dict[slot]['distribution_low_hit_median'] += 1
                    if current_high >= max_range_high_med:
                        stats_dict[slot]['max_range_high_hit_median'] += 1
                    if current_low <= max_range_low_med:
                        stats_dict[slot]['max_range_low_hit_median'] += 1
                    
                    # Check if MEAN-based levels were hit
                    manip_high_hit_mean = current_high >= manipulation_high_mean
                    manip_low_hit_mean = current_low <= manipulation_low_mean
                    dist_high_hit_mean = current_high >= distribution_high_mean
                    dist_low_hit_mean = current_low <= distribution_low_mean
                    
                    if manip_high_hit_mean:
                        stats_dict[slot]['manipulation_high_hit_mean'] += 1
                    if manip_low_hit_mean:
                        stats_dict[slot]['manipulation_low_hit_mean'] += 1
                    if dist_high_hit_mean:
                        stats_dict[slot]['distribution_high_hit_mean'] += 1
                    if dist_low_hit_mean:
                        stats_dict[slot]['distribution_low_hit_mean'] += 1
                    if current_high >= max_range_high_mean:
                        stats_dict[slot]['max_range_high_hit_mean'] += 1
                    if current_low <= max_range_low_mean:
                        stats_dict[slot]['max_range_low_hit_mean'] += 1
                    
                    # Percentile hits (same for both)
                    if current_high >= percentile_75_high_level:
                        stats_dict[slot]['percentile_75_high_hit'] += 1
                    if current_high >= percentile_25_high_level:
                        stats_dict[slot]['percentile_25_high_hit'] += 1
                    if current_low <= percentile_75_low_level:
                        stats_dict[slot]['percentile_75_low_hit'] += 1
                    if current_low <= percentile_25_low_level:
                        stats_dict[slot]['percentile_25_low_hit'] += 1
                    
                    # Level sequencing - MEDIAN
                    if manip_high_hit_med:
                        if dist_high_hit_med:
                            stats_dict[slot]['manip_high_then_dist_high_median'] += 1
                        if manip_low_hit_med:
                            stats_dict[slot]['manip_high_then_reverse_median'] += 1
                    
                    if manip_low_hit_med:
                        if dist_low_hit_med:
                            stats_dict[slot]['manip_low_then_dist_low_median'] += 1
                        if manip_high_hit_med:
                            stats_dict[slot]['manip_low_then_reverse_median'] += 1
                    
                    # Level sequencing - MEAN
                    if manip_high_hit_mean:
                        if dist_high_hit_mean:
                            stats_dict[slot]['manip_high_then_dist_high_mean'] += 1
                        if manip_low_hit_mean:
                            stats_dict[slot]['manip_high_then_reverse_mean'] += 1
                    
                    if manip_low_hit_mean:
                        if dist_low_hit_mean:
                            stats_dict[slot]['manip_low_then_dist_low_mean'] += 1
                        if manip_high_hit_mean:
                            stats_dict[slot]['manip_low_then_reverse_mean'] += 1
                    
                    # Double-sided hits - MEDIAN
                    if manip_high_hit_med and manip_low_hit_med:
                        stats_dict[slot]['both_manip_levels_hit_median'] += 1
                    if dist_high_hit_med and dist_low_hit_med:
                        stats_dict[slot]['both_dist_levels_hit_median'] += 1
                    
                    # Double-sided hits - MEAN
                    if manip_high_hit_mean and manip_low_hit_mean:
                        stats_dict[slot]['both_manip_levels_hit_mean'] += 1
                    if dist_high_hit_mean and dist_low_hit_mean:
                        stats_dict[slot]['both_dist_levels_hit_mean'] += 1
                    
                    # Conditional probabilities - MEDIAN
                    if is_bullish_candle and dist_high_hit_med:
                        stats_dict[slot]['bullish_and_hit_dist_high_median'] += 1
                    if not is_bullish_candle and dist_low_hit_med:
                        stats_dict[slot]['bearish_and_hit_dist_low_median'] += 1
                    if manip_low_hit_med and is_bullish_candle:
                        stats_dict[slot]['manip_low_hit_and_closed_bullish_median'] += 1
                    if manip_high_hit_med and not is_bullish_candle:
                        stats_dict[slot]['manip_high_hit_and_closed_bearish_median'] += 1
                    
                    # Conditional probabilities - MEAN
                    if is_bullish_candle and dist_high_hit_mean:
                        stats_dict[slot]['bullish_and_hit_dist_high_mean'] += 1
                    if not is_bullish_candle and dist_low_hit_mean:
                        stats_dict[slot]['bearish_and_hit_dist_low_mean'] += 1
                    if manip_low_hit_mean and is_bullish_candle:
                        stats_dict[slot]['manip_low_hit_and_closed_bullish_mean'] += 1
                    if manip_high_hit_mean and not is_bullish_candle:
                        stats_dict[slot]['manip_high_hit_and_closed_bearish_mean'] += 1
                    
                    # Check where close is relative to MEDIAN levels
                    if current_close > manipulation_high_med:
                        stats_dict[slot]['close_above_manipulation_high_median'] += 1
                    if current_close < manipulation_low_med:
                        stats_dict[slot]['close_below_manipulation_low_median'] += 1
                    if current_close > distribution_high_med:
                        stats_dict[slot]['close_above_distribution_high_median'] += 1
                    if current_close < distribution_low_med:
                        stats_dict[slot]['close_below_distribution_low_median'] += 1
                    if current_close > max_range_high_med:
                        stats_dict[slot]['close_above_max_range_high_median'] += 1
                    if current_close < max_range_low_med:
                        stats_dict[slot]['close_below_max_range_low_median'] += 1
                    
                    # Check where close is relative to MEAN levels
                    if current_close > manipulation_high_mean:
                        stats_dict[slot]['close_above_manipulation_high_mean'] += 1
                    if current_close < manipulation_low_mean:
                        stats_dict[slot]['close_below_manipulation_low_mean'] += 1
                    if current_close > distribution_high_mean:
                        stats_dict[slot]['close_above_distribution_high_mean'] += 1
                    if current_close < distribution_low_mean:
                        stats_dict[slot]['close_below_distribution_low_mean'] += 1
                    if current_close > max_range_high_mean:
                        stats_dict[slot]['close_above_max_range_high_mean'] += 1
                    if current_close < max_range_low_mean:
                        stats_dict[slot]['close_below_max_range_low_mean'] += 1
                    
                    # Check where close is relative to percentiles
                    if current_close > percentile_75_high_level:
                        stats_dict[slot]['close_above_percentile_75_high'] += 1
                    if current_close < percentile_75_low_level:
                        stats_dict[slot]['close_below_percentile_75_low'] += 1
    
    # Aggregate and process all candle types
    print(f"\nAggregating candles for {timeframe_label}...")
    sys.stdout.flush()
    
    # Hourly
    hourly_groups = data.groupby(['hour_id', 'trading_hour'], sort=False).agg({
        'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last'
    }).reset_index()
    print(f"  ✓ {len(hourly_groups):,} hourly candles")
    process_candles(hourly_groups, hourly_stats, 'trading_hour', 24, "hourly")
    
    # 4H
    four_h_groups = data.groupby(['4h_id', 'trading_4h_slot'], sort=False).agg({
        'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last'
    }).reset_index()
    print(f"  ✓ {len(four_h_groups):,} 4H candles")
    process_candles(four_h_groups, four_hour_stats, 'trading_4h_slot', 6, "4H")
    
    # 6H
    six_h_groups = data.groupby(['6h_id', 'trading_6h_slot'], sort=False).agg({
        'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last'
    }).reset_index()
    print(f"  ✓ {len(six_h_groups):,} 6H candles")
    process_candles(six_h_groups, six_hour_stats, 'trading_6h_slot', 4, "6H")
    
    # 8H
    eight_h_groups = data.groupby(['8h_id', 'trading_8h_slot'], sort=False).agg({
        'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last'
    }).reset_index()
    print(f"  ✓ {len(eight_h_groups):,} 8H candles")
    process_candles(eight_h_groups, eight_hour_stats, 'trading_8h_slot', 3, "8H")
    
    # 12H
    twelve_h_groups = data.groupby(['12h_id', 'trading_12h_slot'], sort=False).agg({
        'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last'
    }).reset_index()
    print(f"  ✓ {len(twelve_h_groups):,} 12H candles")
    process_candles(twelve_h_groups, twelve_hour_stats, 'trading_12h_slot', 2, "12H")
    
    # Daily
    daily_groups = data.groupby(['daily_id'], sort=False).agg({
        'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last'
    }).reset_index()
    print(f"  ✓ {len(daily_groups):,} daily candles")
    process_candles(daily_groups, daily_stats, None, None, "daily")
    
    print(f"✓ {timeframe_label} complete")
    sys.stdout.flush()
    
    return {
        'hourly': hourly_stats,
        '4h': four_hour_stats,
        '6h': six_hour_stats,
        '8h': eight_hour_stats,
        '12h': twelve_hour_stats,
        'daily': daily_stats
    }

# ============================================================
# RUN ANALYSIS FOR ALL TIMEFRAMES
# ============================================================

results = {}

print("\n" + "=" * 80)
print("RUNNING ANALYSIS")
print("=" * 80)
sys.stdout.flush()

results['1y'] = analyze_timeframe(df_1y, "1-YEAR")
results['5y'] = analyze_timeframe(df_5y, "5-YEAR")
results['full'] = analyze_timeframe(df_full, "FULL DATASET")

# ============================================================
# PRINT COMPARISON RESULTS
# ============================================================

def format_hour(trading_hour):
    """Convert trading hour to clock time"""
    if trading_hour < 6:
        clock_hour = trading_hour + 18
    else:
        clock_hour = trading_hour - 6
    
    if clock_hour == 0:
        return "12 AM"
    elif clock_hour < 12:
        return f"{clock_hour} AM"
    elif clock_hour == 12:
        return "12 PM"
    else:
        return f"{clock_hour - 12} PM"

def format_4h_slot(slot):
    slot_times = [
        "18:00-22:00 (6 PM - 10 PM)",
        "22:00-02:00 (10 PM - 2 AM)",
        "02:00-06:00 (2 AM - 6 AM)",
        "06:00-10:00 (6 AM - 10 AM)",
        "10:00-14:00 (10 AM - 2 PM)",
        "14:00-18:00 (2 PM - 6 PM)"
    ]
    return slot_times[slot]

def format_6h_slot(slot):
    slot_times = [
        "18:00-00:00 (6 PM - Midnight)",
        "00:00-06:00 (Midnight - 6 AM)",
        "06:00-12:00 (6 AM - Noon)",
        "12:00-18:00 (Noon - 6 PM)"
    ]
    return slot_times[slot]

def format_8h_slot(slot):
    slot_times = [
        "18:00-02:00 (6 PM - 2 AM)",
        "02:00-10:00 (2 AM - 10 AM)",
        "10:00-18:00 (10 AM - 6 PM)"
    ]
    return slot_times[slot]

def format_12h_slot(slot):
    slot_times = [
        "18:00-06:00 (6 PM - 6 AM)",
        "06:00-18:00 (6 AM - 6 PM)"
    ]
    return slot_times[slot]

print("\n" + "=" * 100)
print("TIMEFRAME COMPARISON RESULTS - ALL STATISTICS")
print("=" * 100)
sys.stdout.flush()

timeframes = ['1y', '5y', 'full']
tf_labels = {'1y': '1-Year', '5y': '5-Year', 'full': 'Full'}

def print_comparison_stats(stats_dict, label, timeframes_list, tf_labels_dict):
    """Print comprehensive comparison statistics"""
    
    print(f"\n{'=' * 100}")
    print(f"{label}")
    print(f"{'=' * 100}")
    
    print(f"\n{'Metric':<50} {'1-Year':<15} {'5-Year':<15} {'Full':<15}")
    print("-" * 100)
    
    # Total candles
    for tf in timeframes_list:
        if isinstance(stats_dict[tf], dict):
            total = stats_dict[tf]['total_candles']
        else:
            total = stats_dict[tf]
        print(f"Total Candles: {tf_labels_dict[tf]:<37} {total:<15,}")
    
    print("\nLEVEL DISTANCES (% from open):")
    print("-" * 100)
    
    for metric_name, metric_key, calc_type in [
        ('Manipulation Wick - Median', 'manip_wick_percentages', 'median'),
        ('Manipulation Wick - Mean', 'manip_wick_percentages', 'mean'),
        ('Manipulation Wick - 25th %ile', 'manip_wick_percentages', 'p25'),
        ('Manipulation Wick - 75th %ile', 'manip_wick_percentages', 'p75'),
        ('Distribution Distance - Median', 'dist_distance_percentages', 'median'),
        ('Distribution Distance - Mean', 'dist_distance_percentages', 'mean'),
        ('Distribution Distance - 25th %ile', 'dist_distance_percentages', 'p25'),
        ('Distribution Distance - 75th %ile', 'dist_distance_percentages', 'p75'),
        ('Max Range - Median', 'max_range_percentages', 'median'),
        ('High Range - 75th %ile', 'high_range_percentages', 'p75'),
        ('High Range - 25th %ile', 'high_range_percentages', 'p25'),
        ('Low Range - 75th %ile', 'low_range_percentages', 'p75'),
        ('Low Range - 25th %ile', 'low_range_percentages', 'p25'),
        ('Total Dist Level (Manip+Dist) - Median', None, 'total_dist')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            if metric_key and len(st[metric_key]) > 0:
                if calc_type == 'median':
                    values.append(f"{np.median(st[metric_key]):.3f}%")
                elif calc_type == 'mean':
                    values.append(f"{np.mean(st[metric_key]):.3f}%")
                elif calc_type == 'p25':
                    values.append(f"{np.percentile(st[metric_key], 25):.3f}%")
                elif calc_type == 'p75':
                    values.append(f"{np.percentile(st[metric_key], 75):.3f}%")
            elif calc_type == 'total_dist':
                manip_med = np.median(st['manip_wick_percentages'])
                dist_med = np.median(st['dist_distance_percentages'])
                values.append(f"{manip_med + dist_med:.3f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nLEVEL HIT RATES - MEDIAN-BASED LEVELS (%):")
    print("-" * 100)
    
    for metric_name, metric_key in [
        ('Manipulation High Hit', 'manipulation_high_hit_median'),
        ('Manipulation Low Hit', 'manipulation_low_hit_median'),
        ('Distribution High Hit', 'distribution_high_hit_median'),
        ('Distribution Low Hit', 'distribution_low_hit_median'),
        ('Max Range High Hit', 'max_range_high_hit_median'),
        ('Max Range Low Hit', 'max_range_low_hit_median'),
        ('Both Manip Levels Hit', 'both_manip_levels_hit_median'),
        ('Both Dist Levels Hit', 'both_dist_levels_hit_median')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            total = st['total_candles']
            if total > 0:
                pct = (st[metric_key] / total * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nLEVEL HIT RATES - MEAN-BASED LEVELS (%):")
    print("-" * 100)
    
    for metric_name, metric_key in [
        ('Manipulation High Hit', 'manipulation_high_hit_mean'),
        ('Manipulation Low Hit', 'manipulation_low_hit_mean'),
        ('Distribution High Hit', 'distribution_high_hit_mean'),
        ('Distribution Low Hit', 'distribution_low_hit_mean'),
        ('Max Range High Hit', 'max_range_high_hit_mean'),
        ('Max Range Low Hit', 'max_range_low_hit_mean'),
        ('Both Manip Levels Hit', 'both_manip_levels_hit_mean'),
        ('Both Dist Levels Hit', 'both_dist_levels_hit_mean')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            total = st['total_candles']
            if total > 0:
                pct = (st[metric_key] / total * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nPERCENTILE-BASED HIT RATES (%):")
    print("-" * 100)
    
    for metric_name, metric_key in [
        ('Percentile 75 High Hit', 'percentile_75_high_hit'),
        ('Percentile 25 High Hit', 'percentile_25_high_hit'),
        ('Percentile 75 Low Hit', 'percentile_75_low_hit'),
        ('Percentile 25 Low Hit', 'percentile_25_low_hit')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            total = st['total_candles']
            if total > 0:
                pct = (st[metric_key] / total * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nCLOSE POSITION RELATIVE TO MEDIAN-BASED LEVELS (%):")
    print("-" * 100)
    
    for metric_name, metric_key in [
        ('Close Above Manipulation High', 'close_above_manipulation_high_median'),
        ('Close Below Manipulation Low', 'close_below_manipulation_low_median'),
        ('Close Above Distribution High', 'close_above_distribution_high_median'),
        ('Close Below Distribution Low', 'close_below_distribution_low_median'),
        ('Close Above Max Range High', 'close_above_max_range_high_median'),
        ('Close Below Max Range Low', 'close_below_max_range_low_median')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            total = st['total_candles']
            if total > 0:
                pct = (st[metric_key] / total * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nCLOSE POSITION RELATIVE TO MEAN-BASED LEVELS (%):")
    print("-" * 100)
    
    for metric_name, metric_key in [
        ('Close Above Manipulation High', 'close_above_manipulation_high_mean'),
        ('Close Below Manipulation Low', 'close_below_manipulation_low_mean'),
        ('Close Above Distribution High', 'close_above_distribution_high_mean'),
        ('Close Below Distribution Low', 'close_below_distribution_low_mean'),
        ('Close Above Max Range High', 'close_above_max_range_high_mean'),
        ('Close Below Max Range Low', 'close_below_max_range_low_mean')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            total = st['total_candles']
            if total > 0:
                pct = (st[metric_key] / total * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nCLOSE POSITION RELATIVE TO PERCENTILES (%):")
    print("-" * 100)
    
    for metric_name, metric_key in [
        ('Close Above Percentile 75 High', 'close_above_percentile_75_high'),
        ('Close Below Percentile 75 Low', 'close_below_percentile_75_low')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            total = st['total_candles']
            if total > 0:
                pct = (st[metric_key] / total * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nDIRECTIONAL BIAS (%):")
    print("-" * 100)
    
    for metric_name, metric_key in [
        ('Bullish Candles', 'bullish_candles'),
        ('Bearish Candles', 'bearish_candles'),
        ('Doji Candles', 'doji_candles')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            total = st['total_candles']
            if total > 0:
                pct = (st[metric_key] / total * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nSTRONG MOVES (%):")
    print("-" * 100)
    
    for metric_name, metric_key in [
        ('Close in Top 25%', 'close_top_25_percent'),
        ('Close in Bottom 25%', 'close_bottom_25_percent')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            total = st['total_candles']
            if total > 0:
                pct = (st[metric_key] / total * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nLEVEL SEQUENCING - MEDIAN-BASED (%):")
    print("-" * 100)
    
    # Manip high sequencing
    print("  After Manipulation High Hit (median):")
    for metric_name, metric_key, denominator_key in [
        ('  → Then Dist High', 'manip_high_then_dist_high_median', 'manipulation_high_hit_median'),
        ('  → Then Reversed', 'manip_high_then_reverse_median', 'manipulation_high_hit_median')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            denom = st[denominator_key]
            if denom > 0:
                pct = (st[metric_key] / denom * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    # Manip low sequencing
    print("  After Manipulation Low Hit (median):")
    for metric_name, metric_key, denominator_key in [
        ('  → Then Dist Low', 'manip_low_then_dist_low_median', 'manipulation_low_hit_median'),
        ('  → Then Reversed', 'manip_low_then_reverse_median', 'manipulation_low_hit_median')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            denom = st[denominator_key]
            if denom > 0:
                pct = (st[metric_key] / denom * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nLEVEL SEQUENCING - MEAN-BASED (%):")
    print("-" * 100)
    
    # Manip high sequencing
    print("  After Manipulation High Hit (mean):")
    for metric_name, metric_key, denominator_key in [
        ('  → Then Dist High', 'manip_high_then_dist_high_mean', 'manipulation_high_hit_mean'),
        ('  → Then Reversed', 'manip_high_then_reverse_mean', 'manipulation_high_hit_mean')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            denom = st[denominator_key]
            if denom > 0:
                pct = (st[metric_key] / denom * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    # Manip low sequencing
    print("  After Manipulation Low Hit (mean):")
    for metric_name, metric_key, denominator_key in [
        ('  → Then Dist Low', 'manip_low_then_dist_low_mean', 'manipulation_low_hit_mean'),
        ('  → Then Reversed', 'manip_low_then_reverse_mean', 'manipulation_low_hit_mean')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            denom = st[denominator_key]
            if denom > 0:
                pct = (st[metric_key] / denom * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nCONDITIONAL PROBABILITIES - MEDIAN-BASED (%):")
    print("-" * 100)
    
    for metric_name, metric_key, denominator_key in [
        ('Given Bullish → Hit Dist High', 'bullish_and_hit_dist_high_median', 'bullish_candles'),
        ('Given Bearish → Hit Dist Low', 'bearish_and_hit_dist_low_median', 'bearish_candles'),
        ('Manip Low Hit & Closed Bullish', 'manip_low_hit_and_closed_bullish_median', 'manipulation_low_hit_median'),
        ('Manip High Hit & Closed Bearish', 'manip_high_hit_and_closed_bearish_median', 'manipulation_high_hit_median')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            denom = st[denominator_key]
            if denom > 0:
                pct = (st[metric_key] / denom * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nCONDITIONAL PROBABILITIES - MEAN-BASED (%):")
    print("-" * 100)
    
    for metric_name, metric_key, denominator_key in [
        ('Given Bullish → Hit Dist High', 'bullish_and_hit_dist_high_mean', 'bullish_candles'),
        ('Given Bearish → Hit Dist Low', 'bearish_and_hit_dist_low_mean', 'bearish_candles'),
        ('Manip Low Hit & Closed Bullish', 'manip_low_hit_and_closed_bullish_mean', 'manipulation_low_hit_mean'),
        ('Manip High Hit & Closed Bearish', 'manip_high_hit_and_closed_bearish_mean', 'manipulation_high_hit_mean')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            denom = st[denominator_key]
            if denom > 0:
                pct = (st[metric_key] / denom * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")

# Print Daily comparison
daily_stats_comparison = {tf: results[tf]['daily'] for tf in timeframes}
print_comparison_stats(daily_stats_comparison, "DAILY CANDLE COMPARISON", timeframes, tf_labels)

# Print 12H comparison
for slot in range(2):
    slot_stats = {tf: results[tf]['12h'][slot] for tf in timeframes}
    label = f"12H SLOT {slot}: {format_12h_slot(slot)}"
    print_comparison_stats(slot_stats, label, timeframes, tf_labels)

# Print 8H comparison
for slot in range(3):
    slot_stats = {tf: results[tf]['8h'][slot] for tf in timeframes}
    label = f"8H SLOT {slot}: {format_8h_slot(slot)}"
    print_comparison_stats(slot_stats, label, timeframes, tf_labels)

# Print 6H comparison
for slot in range(4):
    slot_stats = {tf: results[tf]['6h'][slot] for tf in timeframes}
    label = f"6H SLOT {slot}: {format_6h_slot(slot)}"
    print_comparison_stats(slot_stats, label, timeframes, tf_labels)

# Print 4H comparison
for slot in range(6):
    slot_stats = {tf: results[tf]['4h'][slot] for tf in timeframes}
    label = f"4H SLOT {slot}: {format_4h_slot(slot)}"
    print_comparison_stats(slot_stats, label, timeframes, tf_labels)

# Print Hourly comparison (all 24 hours)
for hour in range(24):
    hour_stats = {tf: results[tf]['hourly'][hour] for tf in timeframes}
    label = f"TRADING HOUR {hour}: {format_hour(hour)}"
    print_comparison_stats(hour_stats, label, timeframes, tf_labels)

print("\n" + "=" * 100)
print("COMPLETE TIMEFRAME COMPARISON ANALYSIS COMPLETE - ALL STATISTICS INCLUDED")
print("=" * 100)
sys.stdout.flush()

STATISTICAL LEVEL ANALYSIS - COMPLETE TIMEFRAME COMPARISON (ALL STATS)

Loading data from: C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv
✓ Loaded 4,974,698 rows

Creating datetime index...
✓ Datetime index created

Filtering to 2013-2025...
✓ Filtered to 4,161,727 rows

Resampling to 5-minute bars...
✓ Resampled to 854,417 5-minute bars

Timeframe cutoffs:
  Full dataset: 2013-01-02 to 2025-03-14
  5-year: 2020-03-15 to 2025-03-14
  1-year: 2024-03-14 to 2025-03-14

Dataset sizes:
  Full: 854,417 bars
  5-year: 353,363 bars
  1-year: 70,648 bars

RUNNING ANALYSIS

ANALYZING: 1-YEAR

Aggregating candles for 1-YEAR...
  ✓ 5,891 hourly candles
  Processing hourly...
  ✓ 1,537 4H candles
  Processing 4H...
  ✓ 1,032 6H candles
  Processing 6H...
  ✓ 829 8H candles
  Processing 8H...
  ✓ 517 12H candles
  Processing 12H...
  ✓ 313 daily candles
  Processing daily...
✓ 1-YEAR complete

ANALYZING: 5-YEAR

Aggregating candles for 5-YEAR...
  ✓ 29,551 hourly candles
  Processing hourly...
 

Manipulation Wick - 25th %ile                      0.036%          0.052%          0.042%         
Manipulation Wick - 75th %ile                      0.166%          0.224%          0.180%         
Distribution Distance - Median                     0.316%          0.406%          0.322%         
Distribution Distance - Mean                       0.404%          0.532%          0.436%         
Distribution Distance - 25th %ile                  0.198%          0.259%          0.194%         
Distribution Distance - 75th %ile                  0.487%          0.659%          0.535%         
Max Range - Median                                 0.330%          0.423%          0.338%         
High Range - 75th %ile                             0.323%          0.443%          0.362%         
High Range - 25th %ile                             0.070%          0.113%          0.088%         
Low Range - 75th %ile                              0.362%          0.458%          0.371%         
Low Range 

Low Range - 25th %ile                              0.032%          0.032%          0.021%         
Total Dist Level (Manip+Dist) - Median             0.139%          0.158%          0.116%         

LEVEL HIT RATES - MEDIAN-BASED LEVELS (%):
----------------------------------------------------------------------------------------------------
Manipulation High Hit                              71.71%          74.07%          69.19%         
Manipulation Low Hit                               72.87%          73.92%          66.06%         
Distribution High Hit                              19.77%          19.58%          29.02%         
Distribution Low Hit                               19.77%          20.67%          27.58%         
Max Range High Hit                                 27.52%          27.24%          31.77%         
Max Range Low Hit                                  27.13%          28.25%          30.17%         
Both Manip Levels Hit                              46.51%      

High Range - 25th %ile                             0.025%          0.030%          0.024%         
Low Range - 75th %ile                              0.103%          0.130%          0.102%         
Low Range - 25th %ile                              0.022%          0.029%          0.021%         
Total Dist Level (Manip+Dist) - Median             0.109%          0.143%          0.113%         

LEVEL HIT RATES - MEDIAN-BASED LEVELS (%):
----------------------------------------------------------------------------------------------------
Manipulation High Hit                              73.26%          74.40%          70.18%         
Manipulation Low Hit                               72.09%          73.09%          67.82%         
Distribution High Hit                              20.16%          21.04%          27.62%         
Distribution Low Hit                               23.64%          21.81%          26.53%         
Max Range High Hit                                 28.29%      

Total Dist Level (Manip+Dist) - Median             0.158%          0.206%          0.163%         

LEVEL HIT RATES - MEDIAN-BASED LEVELS (%):
----------------------------------------------------------------------------------------------------
Manipulation High Hit                              72.09%          71.23%          69.38%         
Manipulation Low Hit                               72.09%          74.17%          69.67%         
Distribution High Hit                              15.50%          15.85%          26.03%         
Distribution Low Hit                               19.38%          20.11%          28.64%         
Max Range High Hit                                 26.74%          25.60%          29.92%         
Max Range Low Hit                                  28.29%          29.16%          32.22%         
Both Manip Levels Hit                              46.90%          47.02%          47.78%         
Both Dist Levels Hit                               1.94%       

Total Dist Level (Manip+Dist) - Median             nan%            0.000%          0.042%         

LEVEL HIT RATES - MEDIAN-BASED LEVELS (%):
----------------------------------------------------------------------------------------------------
Manipulation High Hit                              N/A             100.00%         84.93%         
Manipulation Low Hit                               N/A             100.00%         83.66%         
Distribution High Hit                              N/A             100.00%         27.46%         
Distribution Low Hit                               N/A             100.00%         23.10%         
Max Range High Hit                                 N/A             100.00%         34.51%         
Max Range Low Hit                                  N/A             100.00%         29.44%         
Both Manip Levels Hit                              N/A             100.00%         71.55%         
Both Dist Levels Hit                               N/A         

C:\Users\tammm\.conda\lib\site-packages\numpy\core\fromnumeric.py:3440: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\tammm\.conda\lib\site-packages\numpy\core\_methods.py:189: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


In [3]:
## 60 Candle Lookback with 1y, 5y, and Full Dataset Comparison

# Statistical level hit analysis - EXTENDED VERSION WITH TRADING METRICS AND TIMEFRAME COMPARISON
## 
## This script implements TradingView indicator logic to analyze how often projected levels
## are hit during hourly, 4H, 6H, 8H, 12H, and daily candles based on historical statistical patterns.
##
## TIMEFRAME COMPARISON:
## - Last 1 year
## - Last 5 years
## - Full dataset (2013-2025)
##
## Trading Day: 18:00 (6 PM) to 17:00 (5 PM) next day
##
## LOOKBACK: 60 candles for level calculation
##
## METRICS TRACKED:
## - Directional bias (bullish/bearish %)
## - Strong moves (close in top/bottom 25% of range)
## - Consecutive direction patterns
## - Reversal frequency after hitting levels
## - Level sequencing probabilities
## - Double-sided hits
## - Retracement analysis
## - Timing analysis (early/late hits, formation speed)
## - Hour-to-hour relationships (carryover, setup patterns)
## - Range expansion/contraction
## - Conditional probabilities
##
## Performance Optimization: 
## - Data is resampled to 5-minute bars (from 1-minute) for 5x faster processing
## - This does NOT affect OHLC values for hourly/4H candles (they remain identical)
## - High/low formation order timing has ~5min precision instead of 1min (more than sufficient)

import pandas as pd
import numpy as np
from collections import defaultdict
import sys

print("=" * 80)
print("STATISTICAL LEVEL ANALYSIS - 60 LOOKBACK WITH TIMEFRAME COMPARISON")
print("=" * 80)
sys.stdout.flush()

# Load the data
file_path = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
print(f"\nLoading data from: {file_path}")
sys.stdout.flush()
df = pd.read_csv(file_path)
print(f"✓ Loaded {len(df):,} rows")
sys.stdout.flush()

# Combine Date and Time into a datetime column
print("\nCreating datetime index...")
sys.stdout.flush()
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)
print(f"✓ Datetime index created")
sys.stdout.flush()

# Filter data from 2013 to 2025
print("\nFiltering to 2013-2025...")
sys.stdout.flush()
start_date = pd.to_datetime('2013-01-01')
end_date = pd.to_datetime('2025-12-31')
df = df.loc[start_date:end_date]
print(f"✓ Filtered to {len(df):,} rows")
sys.stdout.flush()

# Resample to 5-minute bars for performance (5x faster, no loss in statistical accuracy)
print("\nResampling to 5-minute bars...")
sys.stdout.flush()
df = df.resample('5min').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last'
}).dropna()
print(f"✓ Resampled to {len(df):,} 5-minute bars")
sys.stdout.flush()

# Define timeframe cutoffs
current_date = df.index.max()
one_year_ago = current_date - pd.Timedelta(days=365)
five_years_ago = current_date - pd.Timedelta(days=365*5)

print(f"\nTimeframe cutoffs:")
print(f"  Full dataset: {df.index.min().date()} to {df.index.max().date()}")
print(f"  5-year: {five_years_ago.date()} to {df.index.max().date()}")
print(f"  1-year: {one_year_ago.date()} to {df.index.max().date()}")
sys.stdout.flush()

# Create filtered datasets
df_1y = df[df.index >= one_year_ago].copy()
df_5y = df[df.index >= five_years_ago].copy()
df_full = df.copy()

print(f"\nDataset sizes:")
print(f"  Full: {len(df_full):,} bars")
print(f"  5-year: {len(df_5y):,} bars")
print(f"  1-year: {len(df_1y):,} bars")
sys.stdout.flush()

# Add NY trading day column (18:00 to 17:59 next day)
df_1y['NY_Trading_Day'] = (df_1y.index - pd.Timedelta(hours=6)).normalize()
df_5y['NY_Trading_Day'] = (df_5y.index - pd.Timedelta(hours=6)).normalize()
df_full['NY_Trading_Day'] = (df_full.index - pd.Timedelta(hours=6)).normalize()

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_trading_hour(timestamp):
    """
    Get the trading hour (0-23) where 18:00 is hour 0 of the trading day
    Returns the hour offset from 18:00 EST
    """
    hour = timestamp.hour
    # Trading day starts at 18:00 (6 PM)
    # 18:00 = 0, 19:00 = 1, ..., 23:00 = 5, 00:00 = 6, ..., 17:00 = 23
    if hour >= 18:
        return hour - 18
    else:
        return hour + 6

def get_4h_slot(timestamp):
    """
    Get the 4H slot (0-5) where slot 0 starts at 18:00
    Slot 0: 18:00-22:00
    Slot 1: 22:00-02:00
    Slot 2: 02:00-06:00
    Slot 3: 06:00-10:00
    Slot 4: 10:00-14:00
    Slot 5: 14:00-18:00
    """
    trading_hour = get_trading_hour(timestamp)
    return trading_hour // 4

def get_6h_slot(timestamp):
    """
    Get the 6H slot (0-3) where slot 0 starts at 18:00
    Slot 0: 18:00-00:00
    Slot 1: 00:00-06:00
    Slot 2: 06:00-12:00
    Slot 3: 12:00-18:00
    """
    trading_hour = get_trading_hour(timestamp)
    return trading_hour // 6

def get_8h_slot(timestamp):
    """
    Get the 8H slot (0-2) where slot 0 starts at 18:00
    Slot 0: 18:00-02:00
    Slot 1: 02:00-10:00
    Slot 2: 10:00-18:00
    """
    trading_hour = get_trading_hour(timestamp)
    return trading_hour // 8

def get_12h_slot(timestamp):
    """
    Get the 12H slot (0-1) where slot 0 starts at 18:00
    Slot 0: 18:00-06:00
    Slot 1: 06:00-18:00
    """
    trading_hour = get_trading_hour(timestamp)
    return trading_hour // 12

def calculate_percentile(arr, percentile):
    """Calculate percentile from array"""
    if len(arr) == 0:
        return np.nan
    return np.percentile(arr, percentile)

def calculate_manipulation_wick(open_price, high, low, close):
    """
    Manipulation wick: the wick against the candle direction
    Bullish candle: open - low
    Bearish candle: high - open
    """
    if close > open_price:
        return open_price - low
    else:
        return high - open_price

def calculate_distribution_distance(open_price, high, low, close):
    """
    Distribution: extension in the candle's direction
    Bullish candle: high - open
    Bearish candle: open - low
    """
    if close > open_price:
        return high - open_price
    else:
        return open_price - low

def calculate_max_range_from_open(open_price, high, low):
    """Maximum distance from open in either direction"""
    return max(abs(high - open_price), abs(open_price - low))

def calculate_high_range_from_open(open_price, high):
    """Distance from open to high"""
    return high - open_price

def calculate_low_range_from_open(open_price, low):
    """Distance from open to low"""
    return open_price - low

def is_bullish(open_price, close):
    """Check if candle is bullish"""
    return close > open_price

def get_candle_range(high, low):
    """Get total candle range"""
    return high - low

def close_position_in_range(open_price, high, low, close):
    """
    Returns where close is in the range:
    1.0 = at high
    0.0 = at low
    0.5 = middle
    """
    candle_range = high - low
    if candle_range == 0:
        return 0.5
    return (close - low) / candle_range

# ============================================================
# ENHANCED DATA STRUCTURES
# ============================================================

def create_stats_dict():
    """Create a statistics dictionary with all metrics"""
    return {
        'total_candles': 0,
        
        # Original metrics - MEDIAN-based
        'high_first': 0,
        'low_first': 0,
        'simultaneous': 0,
        'manipulation_high_hit_median': 0,
        'manipulation_low_hit_median': 0,
        'distribution_high_hit_median': 0,
        'distribution_low_hit_median': 0,
        'max_range_high_hit_median': 0,
        'max_range_low_hit_median': 0,
        
        # Original metrics - MEAN-based
        'manipulation_high_hit_mean': 0,
        'manipulation_low_hit_mean': 0,
        'distribution_high_hit_mean': 0,
        'distribution_low_hit_mean': 0,
        'max_range_high_hit_mean': 0,
        'max_range_low_hit_mean': 0,
        
        # Percentile-based (same for both)
        'percentile_75_high_hit': 0,
        'percentile_25_high_hit': 0,
        'percentile_75_low_hit': 0,
        'percentile_25_low_hit': 0,
        
        # Close position - MEDIAN-based
        'close_above_manipulation_high_median': 0,
        'close_below_manipulation_low_median': 0,
        'close_above_distribution_high_median': 0,
        'close_below_distribution_low_median': 0,
        'close_above_max_range_high_median': 0,
        'close_below_max_range_low_median': 0,
        
        # Close position - MEAN-based
        'close_above_manipulation_high_mean': 0,
        'close_below_manipulation_low_mean': 0,
        'close_above_distribution_high_mean': 0,
        'close_below_distribution_low_mean': 0,
        'close_above_max_range_high_mean': 0,
        'close_below_max_range_low_mean': 0,
        
        # Close position - Percentile-based
        'close_above_percentile_75_high': 0,
        'close_below_percentile_75_low': 0,
        
        # NEW: Directional bias
        'bullish_candles': 0,
        'bearish_candles': 0,
        'doji_candles': 0,  # close within 10% of range from open
        
        # NEW: Strong moves
        'close_top_25_percent': 0,  # Close in top 25% of range
        'close_bottom_25_percent': 0,  # Close in bottom 25% of range
        
        # NEW: Consecutive direction (stored as lists for analysis)
        'candle_directions': [],  # List of True (bullish) / False (bearish)
        
        # NEW: Level sequencing - MEDIAN-based
        'manip_high_then_dist_high_median': 0,
        'manip_high_then_reverse_median': 0,
        'manip_low_then_dist_low_median': 0,
        'manip_low_then_reverse_median': 0,
        
        # NEW: Level sequencing - MEAN-based
        'manip_high_then_dist_high_mean': 0,
        'manip_high_then_reverse_mean': 0,
        'manip_low_then_dist_low_mean': 0,
        'manip_low_then_reverse_mean': 0,
        
        # NEW: Double-sided hits - MEDIAN-based
        'both_manip_levels_hit_median': 0,
        'both_dist_levels_hit_median': 0,
        
        # NEW: Double-sided hits - MEAN-based
        'both_manip_levels_hit_mean': 0,
        'both_dist_levels_hit_mean': 0,
        
        # NEW: Retracement analysis - MEDIAN-based (when manip high is hit)
        'manip_high_retracements_median': [],
        'manip_high_wick_to_wick_ratios_median': [],
        
        # NEW: Retracement analysis - MEAN-based (when manip high is hit)
        'manip_high_retracements_mean': [],
        'manip_high_wick_to_wick_ratios_mean': [],
        
        # NEW: Retracement analysis - MEDIAN-based (when manip low is hit)
        'manip_low_retracements_median': [],
        'manip_low_wick_to_wick_ratios_median': [],
        
        # NEW: Retracement analysis - MEAN-based (when manip low is hit)
        'manip_low_retracements_mean': [],
        'manip_low_wick_to_wick_ratios_mean': [],
        
        # NEW: Timing analysis - MEDIAN-based
        'manip_high_hit_timing_median': [],
        'manip_low_hit_timing_median': [],
        'dist_high_hit_timing_median': [],
        'dist_low_hit_timing_median': [],
        
        # NEW: Timing analysis - MEAN-based
        'manip_high_hit_timing_mean': [],
        'manip_low_hit_timing_mean': [],
        'dist_high_hit_timing_mean': [],
        'dist_low_hit_timing_mean': [],
        
        # NEW: Formation speed - MEDIAN-based (minutes from open)
        'manip_high_hit_minutes_median': [],
        'manip_low_hit_minutes_median': [],
        'dist_high_hit_minutes_median': [],
        'dist_low_hit_minutes_median': [],
        
        # NEW: Formation speed - MEAN-based (minutes from open)
        'manip_high_hit_minutes_mean': [],
        'manip_low_hit_minutes_mean': [],
        'dist_high_hit_minutes_mean': [],
        'dist_low_hit_minutes_mean': [],
        
        # NEW: Range expansion/contraction
        'range_vs_previous': [],  # Ratio of current range to previous candle range
        
        # NEW: Conditional probabilities - MEDIAN-based
        'bullish_and_hit_dist_high_median': 0,
        'bearish_and_hit_dist_low_median': 0,
        'manip_low_hit_and_closed_bullish_median': 0,
        'manip_high_hit_and_closed_bearish_median': 0,
        
        # NEW: Conditional probabilities - MEAN-based
        'bullish_and_hit_dist_high_mean': 0,
        'bearish_and_hit_dist_low_mean': 0,
        'manip_low_hit_and_closed_bullish_mean': 0,
        'manip_high_hit_and_closed_bearish_mean': 0,
    }

def create_historical_dict():
    """Create a historical data dictionary"""
    return {
        'manipulation_wicks': [],
        'distribution_distances': [],
        'max_ranges': [],
        'high_ranges': [],
        'low_ranges': [],
        'opens': [],
        'highs': [],
        'lows': [],
        'closes': [],
        'timestamps': [],
        'candle_ranges': [],  # For expansion/contraction analysis
    }

# ============================================================
# MAIN ANALYSIS FUNCTION FOR A TIMEFRAME
# ============================================================

def analyze_timeframe(df_data, timeframe_label, lookback_period=60, enable_timing_analysis=True):
    """
    Analyze a complete timeframe with 60 candle lookback
    Returns results dictionary with all statistics
    """
    print(f"\n{'=' * 80}")
    print(f"ANALYZING TIMEFRAME: {timeframe_label}")
    print(f"{'=' * 80}")
    sys.stdout.flush()
    
    # Add trading hour and slot columns for efficient grouping
    print(f"Adding trading hour and slot columns for {timeframe_label}...")
    sys.stdout.flush()
    df_data['trading_hour'] = df_data.index.map(get_trading_hour)
    df_data['trading_4h_slot'] = df_data.index.map(get_4h_slot)
    df_data['trading_6h_slot'] = df_data.index.map(get_6h_slot)
    df_data['trading_8h_slot'] = df_data.index.map(get_8h_slot)
    df_data['trading_12h_slot'] = df_data.index.map(get_12h_slot)
    
    # Create unique identifiers for each candle type
    df_data['hour_id'] = df_data['NY_Trading_Day'].astype(str) + '_' + df_data['trading_hour'].astype(str)
    df_data['4h_id'] = df_data['NY_Trading_Day'].astype(str) + '_' + df_data['trading_4h_slot'].astype(str)
    df_data['6h_id'] = df_data['NY_Trading_Day'].astype(str) + '_' + df_data['trading_6h_slot'].astype(str)
    df_data['8h_id'] = df_data['NY_Trading_Day'].astype(str) + '_' + df_data['trading_8h_slot'].astype(str)
    df_data['12h_id'] = df_data['NY_Trading_Day'].astype(str) + '_' + df_data['trading_12h_slot'].astype(str)
    df_data['daily_id'] = df_data['NY_Trading_Day'].astype(str)
    
    print("✓ Columns added")
    sys.stdout.flush()
    
    # Create data structures for all timeframes
    hourly_historical = {hour: create_historical_dict() for hour in range(24)}
    hourly_stats = {hour: create_stats_dict() for hour in range(24)}
    
    four_hour_historical = {slot: create_historical_dict() for slot in range(6)}
    four_hour_stats = {slot: create_stats_dict() for slot in range(6)}
    
    six_hour_historical = {slot: create_historical_dict() for slot in range(4)}
    six_hour_stats = {slot: create_stats_dict() for slot in range(4)}
    
    eight_hour_historical = {slot: create_historical_dict() for slot in range(3)}
    eight_hour_stats = {slot: create_stats_dict() for slot in range(3)}
    
    twelve_hour_historical = {slot: create_historical_dict() for slot in range(2)}
    twelve_hour_stats = {slot: create_stats_dict() for slot in range(2)}
    
    daily_historical = create_historical_dict()
    daily_stats = create_stats_dict()
    
    # ============================================================
    # PHASE 1: COLLECT HISTORICAL DATA
    # ============================================================
    
    print(f"\nPHASE 1: Collecting historical data for {timeframe_label}")
    sys.stdout.flush()
    
    def aggregate_and_store_candles(df, group_cols, id_col, slot_col, historical_dict, candle_label, num_slots=None):
        """
        Generic function to aggregate candles and store in historical dictionary
        """
        print(f"  Aggregating {candle_label} candles...")
        sys.stdout.flush()
        
        if num_slots is None:
            # Daily candles - no slot grouping
            groups = df.groupby(id_col, sort=False).agg({
                'Open': 'first',
                'High': 'max',
                'Low': 'min',
                'Close': 'last'
            }).reset_index()
            print(f"  ✓ Created {len(groups):,} {candle_label} candles")
            sys.stdout.flush()
            
            # Calculate metrics
            groups['manip_wick'] = groups.apply(
                lambda row: calculate_manipulation_wick(row['Open'], row['High'], row['Low'], row['Close']), 
                axis=1
            )
            groups['dist_distance'] = groups.apply(
                lambda row: calculate_distribution_distance(row['Open'], row['High'], row['Low'], row['Close']), 
                axis=1
            )
            groups['max_range'] = groups.apply(
                lambda row: calculate_max_range_from_open(row['Open'], row['High'], row['Low']), 
                axis=1
            )
            groups['high_range'] = groups.apply(
                lambda row: calculate_high_range_from_open(row['Open'], row['High']), 
                axis=1
            )
            groups['low_range'] = groups.apply(
                lambda row: calculate_low_range_from_open(row['Open'], row['Low']), 
                axis=1
            )
            groups['candle_range'] = groups['High'] - groups['Low']
            
            # Store in historical dict
            historical_dict['manipulation_wicks'] = groups['manip_wick'].tolist()
            historical_dict['distribution_distances'] = groups['dist_distance'].tolist()
            historical_dict['max_ranges'] = groups['max_range'].tolist()
            historical_dict['high_ranges'] = groups['high_range'].tolist()
            historical_dict['low_ranges'] = groups['low_range'].tolist()
            historical_dict['opens'] = groups['Open'].tolist()
            historical_dict['highs'] = groups['High'].tolist()
            historical_dict['lows'] = groups['Low'].tolist()
            historical_dict['closes'] = groups['Close'].tolist()
            historical_dict['timestamps'] = groups[id_col].tolist()
            historical_dict['candle_ranges'] = groups['candle_range'].tolist()
            
        else:
            # Multi-slot candles (hourly, 4H, 6H, 8H, 12H)
            groups = df.groupby(group_cols, sort=False).agg({
                'Open': 'first',
                'High': 'max',
                'Low': 'min',
                'Close': 'last'
            }).reset_index()
            print(f"  ✓ Created {len(groups):,} {candle_label} candles")
            sys.stdout.flush()
            
            for slot in range(num_slots):
                slot_candles = groups[groups[slot_col] == slot].copy()
                
                if len(slot_candles) == 0:
                    continue
                
                # Calculate metrics
                slot_candles['manip_wick'] = slot_candles.apply(
                    lambda row: calculate_manipulation_wick(row['Open'], row['High'], row['Low'], row['Close']), 
                    axis=1
                )
                slot_candles['dist_distance'] = slot_candles.apply(
                    lambda row: calculate_distribution_distance(row['Open'], row['High'], row['Low'], row['Close']), 
                    axis=1
                )
                slot_candles['max_range'] = slot_candles.apply(
                    lambda row: calculate_max_range_from_open(row['Open'], row['High'], row['Low']), 
                    axis=1
                )
                slot_candles['high_range'] = slot_candles.apply(
                    lambda row: calculate_high_range_from_open(row['Open'], row['High']), 
                    axis=1
                )
                slot_candles['low_range'] = slot_candles.apply(
                    lambda row: calculate_low_range_from_open(row['Open'], row['Low']), 
                    axis=1
                )
                slot_candles['candle_range'] = slot_candles['High'] - slot_candles['Low']
                
                # Store in historical dict for this slot
                historical_dict[slot]['manipulation_wicks'] = slot_candles['manip_wick'].tolist()
                historical_dict[slot]['distribution_distances'] = slot_candles['dist_distance'].tolist()
                historical_dict[slot]['max_ranges'] = slot_candles['max_range'].tolist()
                historical_dict[slot]['high_ranges'] = slot_candles['high_range'].tolist()
                historical_dict[slot]['low_ranges'] = slot_candles['low_range'].tolist()
                historical_dict[slot]['opens'] = slot_candles['Open'].tolist()
                historical_dict[slot]['highs'] = slot_candles['High'].tolist()
                historical_dict[slot]['lows'] = slot_candles['Low'].tolist()
                historical_dict[slot]['closes'] = slot_candles['Close'].tolist()
                historical_dict[slot]['timestamps'] = slot_candles[id_col].tolist()
                historical_dict[slot]['candle_ranges'] = slot_candles['candle_range'].tolist()
    
    # Aggregate all timeframes
    aggregate_and_store_candles(df_data, ['hour_id', 'trading_hour'], 'hour_id', 'trading_hour', 
                                hourly_historical, "Hourly", 24)
    
    aggregate_and_store_candles(df_data, ['4h_id', 'trading_4h_slot'], '4h_id', 'trading_4h_slot',
                                four_hour_historical, "4H", 6)
    
    aggregate_and_store_candles(df_data, ['6h_id', 'trading_6h_slot'], '6h_id', 'trading_6h_slot',
                                six_hour_historical, "6H", 4)
    
    aggregate_and_store_candles(df_data, ['8h_id', 'trading_8h_slot'], '8h_id', 'trading_8h_slot',
                                eight_hour_historical, "8H", 3)
    
    aggregate_and_store_candles(df_data, ['12h_id', 'trading_12h_slot'], '12h_id', 'trading_12h_slot',
                                twelve_hour_historical, "12H", 2)
    
    aggregate_and_store_candles(df_data, ['daily_id'], 'daily_id', None,
                                daily_historical, "Daily", None)
    
    print(f"✓ Phase 1 complete for {timeframe_label}!")
    sys.stdout.flush()
    
    # ============================================================
    # PHASE 2: BUILD LOOKUP MAPS
    # ============================================================
    
    print(f"\nPHASE 2: Building lookup maps for {timeframe_label}")
    sys.stdout.flush()
    
    if enable_timing_analysis:
        print("Building lookup maps for timing analysis...")
        sys.stdout.flush()
        
        hour_id_to_minute_data = dict(list(df_data.groupby('hour_id')))
        print(f"  ✓ Built {len(hour_id_to_minute_data):,} hourly lookups")
        sys.stdout.flush()
        
        four_h_id_to_minute_data = dict(list(df_data.groupby('4h_id')))
        print(f"  ✓ Built {len(four_h_id_to_minute_data):,} 4H lookups")
        sys.stdout.flush()
        
        six_h_id_to_minute_data = dict(list(df_data.groupby('6h_id')))
        print(f"  ✓ Built {len(six_h_id_to_minute_data):,} 6H lookups")
        sys.stdout.flush()
        
        eight_h_id_to_minute_data = dict(list(df_data.groupby('8h_id')))
        print(f"  ✓ Built {len(eight_h_id_to_minute_data):,} 8H lookups")
        sys.stdout.flush()
        
        twelve_h_id_to_minute_data = dict(list(df_data.groupby('12h_id')))
        print(f"  ✓ Built {len(twelve_h_id_to_minute_data):,} 12H lookups")
        sys.stdout.flush()
        
        daily_id_to_minute_data = dict(list(df_data.groupby('daily_id')))
        print(f"  ✓ Built {len(daily_id_to_minute_data):,} daily lookups")
        sys.stdout.flush()
    else:
        hour_id_to_minute_data = {}
        four_h_id_to_minute_data = {}
        six_h_id_to_minute_data = {}
        eight_h_id_to_minute_data = {}
        twelve_h_id_to_minute_data = {}
        daily_id_to_minute_data = {}
        print("Timing analysis disabled - skipping lookup maps")
    
    print(f"✓ Phase 2 complete for {timeframe_label}!")
    sys.stdout.flush()
    
    # ============================================================
    # PHASE 3: ANALYZE CANDLES WITH ENHANCED METRICS
    # ============================================================
    
    print(f"\nPHASE 3: Analyzing candles for {timeframe_label}")
    if enable_timing_analysis:
        print("Timing analysis: ENABLED")
    else:
        print("Timing analysis: DISABLED (faster)")
    sys.stdout.flush()
    
    def analyze_candles_enhanced(historical_dict, stats_dict, lookback, id_to_minute_data, 
                                 candle_type, timeframe_minutes, num_slots=None):
        """
        Enhanced candle analysis with all new metrics
        """
        
        if num_slots is None:
            # Daily candles - single dict
            slots_to_analyze = [None]
        else:
            # Multi-slot candles
            slots_to_analyze = range(num_slots)
        
        for slot in slots_to_analyze:
            if slot is None:
                hist = historical_dict
                stats = stats_dict
                slot_label = "Daily"
            else:
                hist = historical_dict[slot]
                stats = stats_dict[slot]
                slot_label = f"{candle_type} slot {slot}"
            
            # Need at least lookback_period + 1 candles
            if len(hist['manipulation_wicks']) <= lookback:
                continue
            
            total_candles = len(hist['manipulation_wicks'])
            candles_to_analyze = total_candles - lookback
            print(f"  {slot_label}: analyzing {candles_to_analyze:,} candles... ", end='')
            sys.stdout.flush()
            
            # Process each candle (starting after we have enough lookback)
            for i in range(lookback, len(hist['manipulation_wicks'])):
                stats['total_candles'] += 1
                
                # Get historical data
                lookback_manip = hist['manipulation_wicks'][i-lookback:i]
                lookback_dist = hist['distribution_distances'][i-lookback:i]
                lookback_max = hist['max_ranges'][i-lookback:i]
                lookback_high_ranges = hist['high_ranges'][i-lookback:i]
                lookback_low_ranges = hist['low_ranges'][i-lookback:i]
                
                # Calculate MEDIAN levels
                median_manip = np.median(lookback_manip)
                median_dist = np.median(lookback_dist)
                median_max = np.median(lookback_max)
                
                # Calculate MEAN levels
                mean_manip = np.mean(lookback_manip)
                mean_dist = np.mean(lookback_dist)
                mean_max = np.mean(lookback_max)
                
                # Calculate percentiles
                percentile_75_high = calculate_percentile(lookback_high_ranges, 75)
                percentile_25_high = calculate_percentile(lookback_high_ranges, 25)
                percentile_75_low = calculate_percentile(lookback_low_ranges, 75)
                percentile_25_low = calculate_percentile(lookback_low_ranges, 25)
                
                # Current candle data
                current_open = hist['opens'][i]
                current_high = hist['highs'][i]
                current_low = hist['lows'][i]
                current_close = hist['closes'][i]
                current_range = hist['candle_ranges'][i]
                candle_id = hist['timestamps'][i]
                
                # Project MEDIAN-based levels from current open
                manipulation_high_med = current_open + median_manip
                manipulation_low_med = current_open - median_manip
                distribution_high_med = current_open + median_dist + median_manip
                distribution_low_med = current_open - median_dist - median_manip
                max_range_high_med = current_open + median_max
                max_range_low_med = current_open - median_max
                
                # Project MEAN-based levels from current open
                manipulation_high_mean = current_open + mean_manip
                manipulation_low_mean = current_open - mean_manip
                distribution_high_mean = current_open + mean_dist + mean_manip
                distribution_low_mean = current_open - mean_dist - mean_manip
                max_range_high_mean = current_open + mean_max
                max_range_low_mean = current_open - mean_max
                
                # Percentile levels (same for both)
                percentile_75_high_level = current_open + percentile_75_high
                percentile_25_high_level = current_open + percentile_25_high
                percentile_75_low_level = current_open - percentile_75_low
                percentile_25_low_level = current_open - percentile_25_low
                
                # NEW: Directional bias
                is_bullish_candle = is_bullish(current_open, current_close)
                if is_bullish_candle:
                    stats['bullish_candles'] += 1
                else:
                    stats['bearish_candles'] += 1
                
                # Check for doji (close within 10% of open relative to range)
                if current_range > 0:
                    body_pct = abs(current_close - current_open) / current_range
                    if body_pct < 0.1:
                        stats['doji_candles'] += 1
                
                # NEW: Strong moves (close in top/bottom 25%)
                close_pos = close_position_in_range(current_open, current_high, current_low, current_close)
                if close_pos >= 0.75:
                    stats['close_top_25_percent'] += 1
                elif close_pos <= 0.25:
                    stats['close_bottom_25_percent'] += 1
                
                # NEW: Track direction for consecutive analysis
                stats['candle_directions'].append(is_bullish_candle)
                
                # NEW: Range expansion/contraction
                if i > lookback:  # Need a previous candle
                    prev_range = hist['candle_ranges'][i-1]
                    if prev_range > 0:
                        range_ratio = current_range / prev_range
                        stats['range_vs_previous'].append(range_ratio)
                
                # Check if MEDIAN-based levels were hit
                manip_high_hit_med = current_high >= manipulation_high_med
                manip_low_hit_med = current_low <= manipulation_low_med
                dist_high_hit_med = current_high >= distribution_high_med
                dist_low_hit_med = current_low <= distribution_low_med
                
                if manip_high_hit_med:
                    stats['manipulation_high_hit_median'] += 1
                if manip_low_hit_med:
                    stats['manipulation_low_hit_median'] += 1
                if dist_high_hit_med:
                    stats['distribution_high_hit_median'] += 1
                if dist_low_hit_med:
                    stats['distribution_low_hit_median'] += 1
                if current_high >= max_range_high_med:
                    stats['max_range_high_hit_median'] += 1
                if current_low <= max_range_low_med:
                    stats['max_range_low_hit_median'] += 1
                
                # Check if MEAN-based levels were hit
                manip_high_hit_mean = current_high >= manipulation_high_mean
                manip_low_hit_mean = current_low <= manipulation_low_mean
                dist_high_hit_mean = current_high >= distribution_high_mean
                dist_low_hit_mean = current_low <= distribution_low_mean
                
                if manip_high_hit_mean:
                    stats['manipulation_high_hit_mean'] += 1
                if manip_low_hit_mean:
                    stats['manipulation_low_hit_mean'] += 1
                if dist_high_hit_mean:
                    stats['distribution_high_hit_mean'] += 1
                if dist_low_hit_mean:
                    stats['distribution_low_hit_mean'] += 1
                if current_high >= max_range_high_mean:
                    stats['max_range_high_hit_mean'] += 1
                if current_low <= max_range_low_mean:
                    stats['max_range_low_hit_mean'] += 1
                
                # Check percentile hits (same for both)
                if current_high >= percentile_75_high_level:
                    stats['percentile_75_high_hit'] += 1
                if current_high >= percentile_25_high_level:
                    stats['percentile_25_high_hit'] += 1
                if current_low <= percentile_75_low_level:
                    stats['percentile_75_low_hit'] += 1
                if current_low <= percentile_25_low_level:
                    stats['percentile_25_low_hit'] += 1
                
                # NEW: Level sequencing - MEDIAN-based
                if manip_high_hit_med:
                    if dist_high_hit_med:
                        stats['manip_high_then_dist_high_median'] += 1
                    if manip_low_hit_med:
                        stats['manip_high_then_reverse_median'] += 1
                
                if manip_low_hit_med:
                    if dist_low_hit_med:
                        stats['manip_low_then_dist_low_median'] += 1
                    if manip_high_hit_med:
                        stats['manip_low_then_reverse_median'] += 1
                
                # NEW: Level sequencing - MEAN-based
                if manip_high_hit_mean:
                    if dist_high_hit_mean:
                        stats['manip_high_then_dist_high_mean'] += 1
                    if manip_low_hit_mean:
                        stats['manip_high_then_reverse_mean'] += 1
                
                if manip_low_hit_mean:
                    if dist_low_hit_mean:
                        stats['manip_low_then_dist_low_mean'] += 1
                    if manip_high_hit_mean:
                        stats['manip_low_then_reverse_mean'] += 1
                
                # NEW: Double-sided hits - MEDIAN-based
                if manip_high_hit_med and manip_low_hit_med:
                    stats['both_manip_levels_hit_median'] += 1
                if dist_high_hit_med and dist_low_hit_med:
                    stats['both_dist_levels_hit_median'] += 1
                
                # NEW: Double-sided hits - MEAN-based
                if manip_high_hit_mean and manip_low_hit_mean:
                    stats['both_manip_levels_hit_mean'] += 1
                if dist_high_hit_mean and dist_low_hit_mean:
                    stats['both_dist_levels_hit_mean'] += 1
                
                # NEW: Conditional probabilities - MEDIAN-based
                if is_bullish_candle and dist_high_hit_med:
                    stats['bullish_and_hit_dist_high_median'] += 1
                if not is_bullish_candle and dist_low_hit_med:
                    stats['bearish_and_hit_dist_low_median'] += 1
                if manip_low_hit_med and is_bullish_candle:
                    stats['manip_low_hit_and_closed_bullish_median'] += 1
                if manip_high_hit_med and not is_bullish_candle:
                    stats['manip_high_hit_and_closed_bearish_median'] += 1
                
                # NEW: Conditional probabilities - MEAN-based
                if is_bullish_candle and dist_high_hit_mean:
                    stats['bullish_and_hit_dist_high_mean'] += 1
                if not is_bullish_candle and dist_low_hit_mean:
                    stats['bearish_and_hit_dist_low_mean'] += 1
                if manip_low_hit_mean and is_bullish_candle:
                    stats['manip_low_hit_and_closed_bullish_mean'] += 1
                if manip_high_hit_mean and not is_bullish_candle:
                    stats['manip_high_hit_and_closed_bearish_mean'] += 1
                
                # Check where close is relative to MEDIAN-based levels
                if current_close > manipulation_high_med:
                    stats['close_above_manipulation_high_median'] += 1
                if current_close < manipulation_low_med:
                    stats['close_below_manipulation_low_median'] += 1
                if current_close > distribution_high_med:
                    stats['close_above_distribution_high_median'] += 1
                if current_close < distribution_low_med:
                    stats['close_below_distribution_low_median'] += 1
                if current_close > max_range_high_med:
                    stats['close_above_max_range_high_median'] += 1
                if current_close < max_range_low_med:
                    stats['close_below_max_range_low_median'] += 1
                
                # Check where close is relative to MEAN-based levels
                if current_close > manipulation_high_mean:
                    stats['close_above_manipulation_high_mean'] += 1
                if current_close < manipulation_low_mean:
                    stats['close_below_manipulation_low_mean'] += 1
                if current_close > distribution_high_mean:
                    stats['close_above_distribution_high_mean'] += 1
                if current_close < distribution_low_mean:
                    stats['close_below_distribution_low_mean'] += 1
                if current_close > max_range_high_mean:
                    stats['close_above_max_range_high_mean'] += 1
                if current_close < max_range_low_mean:
                    stats['close_below_max_range_low_mean'] += 1
                
                # Check where close is relative to percentile levels
                if current_close > percentile_75_high_level:
                    stats['close_above_percentile_75_high'] += 1
                if current_close < percentile_75_low_level:
                    stats['close_below_percentile_75_low'] += 1
                
                # NEW: Timing and retracement analysis (requires minute data)
                if enable_timing_analysis and candle_id in id_to_minute_data:
                    candle_data = id_to_minute_data[candle_id]
                    
                    if not candle_data.empty:
                        candle_start_time = candle_data.index[0]
                        candle_end_time = candle_data.index[-1]
                        candle_duration = (candle_end_time - candle_start_time).total_seconds() / 60  # in minutes
                        
                        # Find when high occurred
                        high_rows = candle_data[candle_data['High'] == current_high]
                        if not high_rows.empty:
                            high_time = high_rows.index[0]
                            
                            # Find when low occurred
                            low_rows = candle_data[candle_data['Low'] == current_low]
                            if not low_rows.empty:
                                low_time = low_rows.index[0]
                                
                                # Formation order
                                if high_time < low_time:
                                    stats['high_first'] += 1
                                elif low_time < high_time:
                                    stats['low_first'] += 1
                                else:
                                    stats['simultaneous'] += 1
                        
                        # NEW: Timing analysis for MEDIAN-based level hits
                        if manip_high_hit_med:
                            # Find when manipulation high was hit
                            manip_high_hit_rows = candle_data[candle_data['High'] >= manipulation_high_med]
                            if not manip_high_hit_rows.empty:
                                hit_time = manip_high_hit_rows.index[0]
                                minutes_from_open = (hit_time - candle_start_time).total_seconds() / 60
                                stats['manip_high_hit_minutes_median'].append(minutes_from_open)
                                
                                if candle_duration > 0:
                                    timing_pct = minutes_from_open / candle_duration
                                    stats['manip_high_hit_timing_median'].append(timing_pct)
                                
                                # Retracement analysis
                                retracement = current_high - current_low
                                stats['manip_high_retracements_median'].append(retracement)
                                
                                # Wick-to-wick ratio
                                if manipulation_low_med < current_low:
                                    wick_to_wick_distance = manipulation_high_med - manipulation_low_med
                                    actual_retracement = current_high - current_low
                                    if wick_to_wick_distance > 0:
                                        ratio = actual_retracement / wick_to_wick_distance
                                        stats['manip_high_wick_to_wick_ratios_median'].append(ratio)
                        
                        if manip_low_hit_med:
                            manip_low_hit_rows = candle_data[candle_data['Low'] <= manipulation_low_med]
                            if not manip_low_hit_rows.empty:
                                hit_time = manip_low_hit_rows.index[0]
                                minutes_from_open = (hit_time - candle_start_time).total_seconds() / 60
                                stats['manip_low_hit_minutes_median'].append(minutes_from_open)
                                
                                if candle_duration > 0:
                                    timing_pct = minutes_from_open / candle_duration
                                    stats['manip_low_hit_timing_median'].append(timing_pct)
                                
                                # Retracement analysis
                                retracement = current_high - current_low
                                stats['manip_low_retracements_median'].append(retracement)
                                
                                # Wick-to-wick ratio
                                if manipulation_high_med > current_high:
                                    wick_to_wick_distance = manipulation_high_med - manipulation_low_med
                                    actual_retracement = current_high - current_low
                                    if wick_to_wick_distance > 0:
                                        ratio = actual_retracement / wick_to_wick_distance
                                        stats['manip_low_wick_to_wick_ratios_median'].append(ratio)
                        
                        if dist_high_hit_med:
                            dist_high_hit_rows = candle_data[candle_data['High'] >= distribution_high_med]
                            if not dist_high_hit_rows.empty:
                                hit_time = dist_high_hit_rows.index[0]
                                minutes_from_open = (hit_time - candle_start_time).total_seconds() / 60
                                stats['dist_high_hit_minutes_median'].append(minutes_from_open)
                                
                                if candle_duration > 0:
                                    timing_pct = minutes_from_open / candle_duration
                                    stats['dist_high_hit_timing_median'].append(timing_pct)
                        
                        if dist_low_hit_med:
                            dist_low_hit_rows = candle_data[candle_data['Low'] <= distribution_low_med]
                            if not dist_low_hit_rows.empty:
                                hit_time = dist_low_hit_rows.index[0]
                                minutes_from_open = (hit_time - candle_start_time).total_seconds() / 60
                                stats['dist_low_hit_minutes_median'].append(minutes_from_open)
                                
                                if candle_duration > 0:
                                    timing_pct = minutes_from_open / candle_duration
                                    stats['dist_low_hit_timing_median'].append(timing_pct)
                        
                        # NEW: Timing analysis for MEAN-based level hits
                        if manip_high_hit_mean:
                            # Find when manipulation high was hit
                            manip_high_hit_rows = candle_data[candle_data['High'] >= manipulation_high_mean]
                            if not manip_high_hit_rows.empty:
                                hit_time = manip_high_hit_rows.index[0]
                                minutes_from_open = (hit_time - candle_start_time).total_seconds() / 60
                                stats['manip_high_hit_minutes_mean'].append(minutes_from_open)
                                
                                if candle_duration > 0:
                                    timing_pct = minutes_from_open / candle_duration
                                    stats['manip_high_hit_timing_mean'].append(timing_pct)
                                
                                # Retracement analysis
                                retracement = current_high - current_low
                                stats['manip_high_retracements_mean'].append(retracement)
                                
                                # Wick-to-wick ratio
                                if manipulation_low_mean < current_low:
                                    wick_to_wick_distance = manipulation_high_mean - manipulation_low_mean
                                    actual_retracement = current_high - current_low
                                    if wick_to_wick_distance > 0:
                                        ratio = actual_retracement / wick_to_wick_distance
                                        stats['manip_high_wick_to_wick_ratios_mean'].append(ratio)
                        
                        if manip_low_hit_mean:
                            manip_low_hit_rows = candle_data[candle_data['Low'] <= manipulation_low_mean]
                            if not manip_low_hit_rows.empty:
                                hit_time = manip_low_hit_rows.index[0]
                                minutes_from_open = (hit_time - candle_start_time).total_seconds() / 60
                                stats['manip_low_hit_minutes_mean'].append(minutes_from_open)
                                
                                if candle_duration > 0:
                                    timing_pct = minutes_from_open / candle_duration
                                    stats['manip_low_hit_timing_mean'].append(timing_pct)
                                
                                # Retracement analysis
                                retracement = current_high - current_low
                                stats['manip_low_retracements_mean'].append(retracement)
                                
                                # Wick-to-wick ratio
                                if manipulation_high_mean > current_high:
                                    wick_to_wick_distance = manipulation_high_mean - manipulation_low_mean
                                    actual_retracement = current_high - current_low
                                    if wick_to_wick_distance > 0:
                                        ratio = actual_retracement / wick_to_wick_distance
                                        stats['manip_low_wick_to_wick_ratios_mean'].append(ratio)
                        
                        if dist_high_hit_mean:
                            dist_high_hit_rows = candle_data[candle_data['High'] >= distribution_high_mean]
                            if not dist_high_hit_rows.empty:
                                hit_time = dist_high_hit_rows.index[0]
                                minutes_from_open = (hit_time - candle_start_time).total_seconds() / 60
                                stats['dist_high_hit_minutes_mean'].append(minutes_from_open)
                                
                                if candle_duration > 0:
                                    timing_pct = minutes_from_open / candle_duration
                                    stats['dist_high_hit_timing_mean'].append(timing_pct)
                        
                        if dist_low_hit_mean:
                            dist_low_hit_rows = candle_data[candle_data['Low'] <= distribution_low_mean]
                            if not dist_low_hit_rows.empty:
                                hit_time = dist_low_hit_rows.index[0]
                                minutes_from_open = (hit_time - candle_start_time).total_seconds() / 60
                                stats['dist_low_hit_minutes_mean'].append(minutes_from_open)
                                
                                if candle_duration > 0:
                                    timing_pct = minutes_from_open / candle_duration
                                    stats['dist_low_hit_timing_mean'].append(timing_pct)
            
            print("✓")
            sys.stdout.flush()
    
    # Analyze all timeframes with enhanced metrics
    print(f"\n  Analyzing hourly candles...")
    sys.stdout.flush()
    analyze_candles_enhanced(hourly_historical, hourly_stats, lookback_period, 
                            hour_id_to_minute_data, "Hourly", 60, 24)
    
    print(f"\n  Analyzing 4H candles...")
    sys.stdout.flush()
    analyze_candles_enhanced(four_hour_historical, four_hour_stats, lookback_period,
                            four_h_id_to_minute_data, "4H", 240, 6)
    
    print(f"\n  Analyzing 6H candles...")
    sys.stdout.flush()
    analyze_candles_enhanced(six_hour_historical, six_hour_stats, lookback_period,
                            six_h_id_to_minute_data, "6H", 360, 4)
    
    print(f"\n  Analyzing 8H candles...")
    sys.stdout.flush()
    analyze_candles_enhanced(eight_hour_historical, eight_hour_stats, lookback_period,
                            eight_h_id_to_minute_data, "8H", 480, 3)
    
    print(f"\n  Analyzing 12H candles...")
    sys.stdout.flush()
    analyze_candles_enhanced(twelve_hour_historical, twelve_hour_stats, lookback_period,
                            twelve_h_id_to_minute_data, "12H", 720, 2)
    
    print(f"\n  Analyzing daily candles...")
    sys.stdout.flush()
    analyze_candles_enhanced(daily_historical, daily_stats, lookback_period,
                            daily_id_to_minute_data, "Daily", 1380, None)
    
    print(f"\n✓ Phase 3 complete for {timeframe_label}!")
    sys.stdout.flush()
    
    # Return all results
    return {
        'hourly': hourly_stats,
        '4h': four_hour_stats,
        '6h': six_hour_stats,
        '8h': eight_hour_stats,
        '12h': twelve_hour_stats,
        'daily': daily_stats
    }

# ============================================================
# RUN ANALYSIS FOR ALL TIMEFRAMES
# ============================================================

lookback_period = 60
enable_timing_analysis = True  # Set to False for much faster processing (skips timing metrics)

print("\n" + "=" * 80)
print("RUNNING ANALYSIS FOR ALL TIMEFRAMES")
print(f"Lookback period: {lookback_period} candles")
print(f"Timing analysis: {'ENABLED' if enable_timing_analysis else 'DISABLED'}")
print("=" * 80)
sys.stdout.flush()

results = {}

results['1y'] = analyze_timeframe(df_1y, "1-YEAR", lookback_period, enable_timing_analysis)
results['5y'] = analyze_timeframe(df_5y, "5-YEAR", lookback_period, enable_timing_analysis)
results['full'] = analyze_timeframe(df_full, "FULL DATASET", lookback_period, enable_timing_analysis)

# ============================================================
# PRINT COMPARISON RESULTS
# ============================================================

def format_hour(trading_hour):
    """Convert trading hour to clock time"""
    if trading_hour < 6:
        clock_hour = trading_hour + 18
    else:
        clock_hour = trading_hour - 6
    
    if clock_hour == 0:
        return "12 AM"
    elif clock_hour < 12:
        return f"{clock_hour} AM"
    elif clock_hour == 12:
        return "12 PM"
    else:
        return f"{clock_hour - 12} PM"

def format_4h_slot(slot):
    slot_times = [
        "18:00-22:00 (6 PM - 10 PM)",
        "22:00-02:00 (10 PM - 2 AM)",
        "02:00-06:00 (2 AM - 6 AM)",
        "06:00-10:00 (6 AM - 10 AM)",
        "10:00-14:00 (10 AM - 2 PM)",
        "14:00-18:00 (2 PM - 6 PM)"
    ]
    return slot_times[slot]

def format_6h_slot(slot):
    slot_times = [
        "18:00-00:00 (6 PM - Midnight)",
        "00:00-06:00 (Midnight - 6 AM)",
        "06:00-12:00 (6 AM - Noon)",
        "12:00-18:00 (Noon - 6 PM)"
    ]
    return slot_times[slot]

def format_8h_slot(slot):
    slot_times = [
        "18:00-02:00 (6 PM - 2 AM)",
        "02:00-10:00 (2 AM - 10 AM)",
        "10:00-18:00 (10 AM - 6 PM)"
    ]
    return slot_times[slot]

def format_12h_slot(slot):
    slot_times = [
        "18:00-06:00 (6 PM - 6 AM)",
        "06:00-18:00 (6 AM - 6 PM)"
    ]
    return slot_times[slot]

print("\n" + "=" * 100)
print("TIMEFRAME COMPARISON RESULTS - ALL STATISTICS")
print(f"Lookback Period: {lookback_period} candles")
print("=" * 100)
sys.stdout.flush()

timeframes = ['1y', '5y', 'full']
tf_labels = {'1y': '1-Year', '5y': '5-Year', 'full': 'Full'}

def print_comparison_stats(stats_dict, label, timeframes_list, tf_labels_dict):
    """Print comprehensive comparison statistics"""
    
    print(f"\n{'=' * 100}")
    print(f"{label}")
    print(f"{'=' * 100}")
    
    print(f"\n{'Metric':<50} {'1-Year':<15} {'5-Year':<15} {'Full':<15}")
    print("-" * 100)
    
    # Total candles
    for tf in timeframes_list:
        if isinstance(stats_dict[tf], dict):
            total = stats_dict[tf]['total_candles']
        else:
            total = stats_dict[tf]
        print(f"Total Candles: {tf_labels_dict[tf]:<37} {total:<15,}")
    
    print("\nDIRECTIONAL BIAS (%):")
    print("-" * 100)
    
    for metric_name, metric_key in [
        ('Bullish Candles', 'bullish_candles'),
        ('Bearish Candles', 'bearish_candles'),
        ('Doji Candles', 'doji_candles')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            total = st['total_candles']
            if total > 0:
                pct = (st[metric_key] / total * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nSTRONG MOVES (%):")
    print("-" * 100)
    
    for metric_name, metric_key in [
        ('Close in Top 25%', 'close_top_25_percent'),
        ('Close in Bottom 25%', 'close_bottom_25_percent')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            total = st['total_candles']
            if total > 0:
                pct = (st[metric_key] / total * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nFORMATION ORDER (%):")
    print("-" * 100)
    
    for metric_name, metric_key in [
        ('High First', 'high_first'),
        ('Low First', 'low_first'),
        ('Simultaneous', 'simultaneous')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            total = st['total_candles']
            if total > 0:
                pct = (st[metric_key] / total * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nLEVEL HIT RATES - MEDIAN-BASED (%):")
    print("-" * 100)
    
    for metric_name, metric_key in [
        ('Manipulation High Hit', 'manipulation_high_hit_median'),
        ('Manipulation Low Hit', 'manipulation_low_hit_median'),
        ('Distribution High Hit', 'distribution_high_hit_median'),
        ('Distribution Low Hit', 'distribution_low_hit_median'),
        ('Max Range High Hit', 'max_range_high_hit_median'),
        ('Max Range Low Hit', 'max_range_low_hit_median'),
        ('Both Manip Levels Hit', 'both_manip_levels_hit_median'),
        ('Both Dist Levels Hit', 'both_dist_levels_hit_median')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            total = st['total_candles']
            if total > 0:
                pct = (st[metric_key] / total * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nLEVEL HIT RATES - MEAN-BASED (%):")
    print("-" * 100)
    
    for metric_name, metric_key in [
        ('Manipulation High Hit', 'manipulation_high_hit_mean'),
        ('Manipulation Low Hit', 'manipulation_low_hit_mean'),
        ('Distribution High Hit', 'distribution_high_hit_mean'),
        ('Distribution Low Hit', 'distribution_low_hit_mean'),
        ('Max Range High Hit', 'max_range_high_hit_mean'),
        ('Max Range Low Hit', 'max_range_low_hit_mean'),
        ('Both Manip Levels Hit', 'both_manip_levels_hit_mean'),
        ('Both Dist Levels Hit', 'both_dist_levels_hit_mean')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            total = st['total_candles']
            if total > 0:
                pct = (st[metric_key] / total * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nPERCENTILE-BASED HIT RATES (%):")
    print("-" * 100)
    
    for metric_name, metric_key in [
        ('Percentile 75 High Hit', 'percentile_75_high_hit'),
        ('Percentile 25 High Hit', 'percentile_25_high_hit'),
        ('Percentile 75 Low Hit', 'percentile_75_low_hit'),
        ('Percentile 25 Low Hit', 'percentile_25_low_hit')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            total = st['total_candles']
            if total > 0:
                pct = (st[metric_key] / total * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nCLOSE POSITION RELATIVE TO MEDIAN-BASED LEVELS (%):")
    print("-" * 100)
    
    for metric_name, metric_key in [
        ('Close Above Manipulation High', 'close_above_manipulation_high_median'),
        ('Close Below Manipulation Low', 'close_below_manipulation_low_median'),
        ('Close Above Distribution High', 'close_above_distribution_high_median'),
        ('Close Below Distribution Low', 'close_below_distribution_low_median'),
        ('Close Above Max Range High', 'close_above_max_range_high_median'),
        ('Close Below Max Range Low', 'close_below_max_range_low_median')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            total = st['total_candles']
            if total > 0:
                pct = (st[metric_key] / total * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nCLOSE POSITION RELATIVE TO MEAN-BASED LEVELS (%):")
    print("-" * 100)
    
    for metric_name, metric_key in [
        ('Close Above Manipulation High', 'close_above_manipulation_high_mean'),
        ('Close Below Manipulation Low', 'close_below_manipulation_low_mean'),
        ('Close Above Distribution High', 'close_above_distribution_high_mean'),
        ('Close Below Distribution Low', 'close_below_distribution_low_mean'),
        ('Close Above Max Range High', 'close_above_max_range_high_mean'),
        ('Close Below Max Range Low', 'close_below_max_range_low_mean')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            total = st['total_candles']
            if total > 0:
                pct = (st[metric_key] / total * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nCLOSE POSITION RELATIVE TO PERCENTILES (%):")
    print("-" * 100)
    
    for metric_name, metric_key in [
        ('Close Above Percentile 75 High', 'close_above_percentile_75_high'),
        ('Close Below Percentile 75 Low', 'close_below_percentile_75_low')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            total = st['total_candles']
            if total > 0:
                pct = (st[metric_key] / total * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
   
    print("\nLEVEL SEQUENCING - MEDIAN-BASED (%):")
    print("-" * 100)
    
    # Manip high sequencing - MEDIAN
    print("  After Manipulation High Hit (median):")
    for metric_name, metric_key, denominator_key in [
        ('  → Then Dist High', 'manip_high_then_dist_high_median', 'manipulation_high_hit_median'),
        ('  → Then Reversed', 'manip_high_then_reverse_median', 'manipulation_high_hit_median')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            denom = st[denominator_key]
            if denom > 0:
                pct = (st[metric_key] / denom * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    # Manip low sequencing - MEDIAN
    print("  After Manipulation Low Hit (median):")
    for metric_name, metric_key, denominator_key in [
        ('  → Then Dist Low', 'manip_low_then_dist_low_median', 'manipulation_low_hit_median'),
        ('  → Then Reversed', 'manip_low_then_reverse_median', 'manipulation_low_hit_median')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            denom = st[denominator_key]
            if denom > 0:
                pct = (st[metric_key] / denom * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nLEVEL SEQUENCING - MEAN-BASED (%):")
    print("-" * 100)
    
    # Manip high sequencing - MEAN
    print("  After Manipulation High Hit (mean):")
    for metric_name, metric_key, denominator_key in [
        ('  → Then Dist High', 'manip_high_then_dist_high_mean', 'manipulation_high_hit_mean'),
        ('  → Then Reversed', 'manip_high_then_reverse_mean', 'manipulation_high_hit_mean')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            denom = st[denominator_key]
            if denom > 0:
                pct = (st[metric_key] / denom * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    # Manip low sequencing - MEAN
    print("  After Manipulation Low Hit (mean):")
    for metric_name, metric_key, denominator_key in [
        ('  → Then Dist Low', 'manip_low_then_dist_low_mean', 'manipulation_low_hit_mean'),
        ('  → Then Reversed', 'manip_low_then_reverse_mean', 'manipulation_low_hit_mean')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            denom = st[denominator_key]
            if denom > 0:
                pct = (st[metric_key] / denom * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nCONDITIONAL PROBABILITIES - MEDIAN-BASED (%):")
    print("-" * 100)
    
    for metric_name, metric_key, denominator_key in [
        ('Given Bullish → Hit Dist High', 'bullish_and_hit_dist_high_median', 'bullish_candles'),
        ('Given Bearish → Hit Dist Low', 'bearish_and_hit_dist_low_median', 'bearish_candles'),
        ('Manip Low Hit & Closed Bullish', 'manip_low_hit_and_closed_bullish_median', 'manipulation_low_hit_median'),
        ('Manip High Hit & Closed Bearish', 'manip_high_hit_and_closed_bearish_median', 'manipulation_high_hit_median')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            denom = st[denominator_key]
            if denom > 0:
                pct = (st[metric_key] / denom * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    print("\nCONDITIONAL PROBABILITIES - MEAN-BASED (%):")
    print("-" * 100)
    
    for metric_name, metric_key, denominator_key in [
        ('Given Bullish → Hit Dist High', 'bullish_and_hit_dist_high_mean', 'bullish_candles'),
        ('Given Bearish → Hit Dist Low', 'bearish_and_hit_dist_low_mean', 'bearish_candles'),
        ('Manip Low Hit & Closed Bullish', 'manip_low_hit_and_closed_bullish_mean', 'manipulation_low_hit_mean'),
        ('Manip High Hit & Closed Bearish', 'manip_high_hit_and_closed_bearish_mean', 'manipulation_high_hit_mean')
    ]:
        values = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            denom = st[denominator_key]
            if denom > 0:
                pct = (st[metric_key] / denom * 100)
                values.append(f"{pct:.2f}%")
            else:
                values.append("N/A")
        print(f"{metric_name:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
    
    if enable_timing_analysis:
        print("\nRETRACEMENT ANALYSIS - MANIP HIGH - MEDIAN-BASED (Median):")
        print("-" * 100)
        
        values_retrace = []
        values_ratio = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            if len(st['manip_high_retracements_median']) > 0:
                values_retrace.append(f"{np.median(st['manip_high_retracements_median']):.2f}")
            else:
                values_retrace.append("N/A")
            
            if len(st['manip_high_wick_to_wick_ratios_median']) > 0:
                values_ratio.append(f"{np.median(st['manip_high_wick_to_wick_ratios_median'])*100:.2f}%")
            else:
                values_ratio.append("N/A")
        
        print(f"{'Median Retracement (points)':<50} {values_retrace[0]:<15} {values_retrace[1]:<15} {values_retrace[2]:<15}")
        print(f"{'Median Wick-to-Wick Ratio':<50} {values_ratio[0]:<15} {values_ratio[1]:<15} {values_ratio[2]:<15}")
        
        print("\nRETRACEMENT ANALYSIS - MANIP HIGH - MEAN-BASED (Median):")
        print("-" * 100)
        
        values_retrace = []
        values_ratio = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            if len(st['manip_high_retracements_mean']) > 0:
                values_retrace.append(f"{np.median(st['manip_high_retracements_mean']):.2f}")
            else:
                values_retrace.append("N/A")
            
            if len(st['manip_high_wick_to_wick_ratios_mean']) > 0:
                values_ratio.append(f"{np.median(st['manip_high_wick_to_wick_ratios_mean'])*100:.2f}%")
            else:
                values_ratio.append("N/A")
        
        print(f"{'Median Retracement (points)':<50} {values_retrace[0]:<15} {values_retrace[1]:<15} {values_retrace[2]:<15}")
        print(f"{'Median Wick-to-Wick Ratio':<50} {values_ratio[0]:<15} {values_ratio[1]:<15} {values_ratio[2]:<15}")
        
        print("\nRETRACEMENT ANALYSIS - MANIP LOW - MEDIAN-BASED (Median):")
        print("-" * 100)
        
        values_retrace = []
        values_ratio = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            if len(st['manip_low_retracements_median']) > 0:
                values_retrace.append(f"{np.median(st['manip_low_retracements_median']):.2f}")
            else:
                values_retrace.append("N/A")
            
            if len(st['manip_low_wick_to_wick_ratios_median']) > 0:
                values_ratio.append(f"{np.median(st['manip_low_wick_to_wick_ratios_median'])*100:.2f}%")
            else:
                values_ratio.append("N/A")
        
        print(f"{'Median Retracement (points)':<50} {values_retrace[0]:<15} {values_retrace[1]:<15} {values_retrace[2]:<15}")
        print(f"{'Median Wick-to-Wick Ratio':<50} {values_ratio[0]:<15} {values_ratio[1]:<15} {values_ratio[2]:<15}")
        
        print("\nRETRACEMENT ANALYSIS - MANIP LOW - MEAN-BASED (Median):")
        print("-" * 100)
        
        values_retrace = []
        values_ratio = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            if len(st['manip_low_retracements_mean']) > 0:
                values_retrace.append(f"{np.median(st['manip_low_retracements_mean']):.2f}")
            else:
                values_retrace.append("N/A")
            
            if len(st['manip_low_wick_to_wick_ratios_mean']) > 0:
                values_ratio.append(f"{np.median(st['manip_low_wick_to_wick_ratios_mean'])*100:.2f}%")
            else:
                values_ratio.append("N/A")
        
        print(f"{'Median Retracement (points)':<50} {values_retrace[0]:<15} {values_retrace[1]:<15} {values_retrace[2]:<15}")
        print(f"{'Median Wick-to-Wick Ratio':<50} {values_ratio[0]:<15} {values_ratio[1]:<15} {values_ratio[2]:<15}")
        
        print("\nTIMING - MANIP HIGH HITS - MEDIAN-BASED:")
        print("-" * 100)
        
        for timing_cat in ['Early (≤25%)', 'Middle (25-75%)', 'Late (>75%)']:
            values = []
            for tf in timeframes_list:
                st = stats_dict[tf]
                if len(st['manip_high_hit_timing_median']) > 0:
                    if timing_cat == 'Early (≤25%)':
                        count = sum(1 for t in st['manip_high_hit_timing_median'] if t <= 0.25)
                    elif timing_cat == 'Middle (25-75%)':
                        count = sum(1 for t in st['manip_high_hit_timing_median'] if 0.25 < t <= 0.75)
                    else:
                        count = sum(1 for t in st['manip_high_hit_timing_median'] if t > 0.75)
                    pct = (count / len(st['manip_high_hit_timing_median']) * 100)
                    values.append(f"{pct:.2f}%")
                else:
                    values.append("N/A")
            print(f"{timing_cat:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
        
        values_avg = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            if len(st['manip_high_hit_minutes_median']) > 0:
                values_avg.append(f"{np.mean(st['manip_high_hit_minutes_median']):.1f}m")
            else:
                values_avg.append("N/A")
        print(f"{'Avg Time to Hit':<50} {values_avg[0]:<15} {values_avg[1]:<15} {values_avg[2]:<15}")
        
        print("\nTIMING - MANIP HIGH HITS - MEAN-BASED:")
        print("-" * 100)
        
        for timing_cat in ['Early (≤25%)', 'Middle (25-75%)', 'Late (>75%)']:
            values = []
            for tf in timeframes_list:
                st = stats_dict[tf]
                if len(st['manip_high_hit_timing_mean']) > 0:
                    if timing_cat == 'Early (≤25%)':
                        count = sum(1 for t in st['manip_high_hit_timing_mean'] if t <= 0.25)
                    elif timing_cat == 'Middle (25-75%)':
                        count = sum(1 for t in st['manip_high_hit_timing_mean'] if 0.25 < t <= 0.75)
                    else:
                        count = sum(1 for t in st['manip_high_hit_timing_mean'] if t > 0.75)
                    pct = (count / len(st['manip_high_hit_timing_mean']) * 100)
                    values.append(f"{pct:.2f}%")
                else:
                    values.append("N/A")
            print(f"{timing_cat:<50} {values[0]:<15} {values[1]:<15} {values[2]:<15}")
        
        values_avg = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            if len(st['manip_high_hit_minutes_mean']) > 0:
                values_avg.append(f"{np.mean(st['manip_high_hit_minutes_mean']):.1f}m")
            else:
                values_avg.append("N/A")
        print(f"{'Avg Time to Hit':<50} {values_avg[0]:<15} {values_avg[1]:<15} {values_avg[2]:<15}")
        
        print("\nRANGE VS PREVIOUS (Median):")
        print("-" * 100)
        
        values_ratio = []
        for tf in timeframes_list:
            st = stats_dict[tf]
            if len(st['range_vs_previous']) > 0:
                values_ratio.append(f"{np.median(st['range_vs_previous']):.2f}x")
            else:
                values_ratio.append("N/A")
        print(f"{'Median Ratio':<50} {values_ratio[0]:<15} {values_ratio[1]:<15} {values_ratio[2]:<15}")

# Print Daily comparison
daily_stats_comparison = {tf: results[tf]['daily'] for tf in timeframes}
print_comparison_stats(daily_stats_comparison, "DAILY CANDLE COMPARISON", timeframes, tf_labels)

# Print 12H comparison
for slot in range(2):
    slot_stats = {tf: results[tf]['12h'][slot] for tf in timeframes}
    label = f"12H SLOT {slot}: {format_12h_slot(slot)}"
    print_comparison_stats(slot_stats, label, timeframes, tf_labels)

# Print 8H comparison
for slot in range(3):
    slot_stats = {tf: results[tf]['8h'][slot] for tf in timeframes}
    label = f"8H SLOT {slot}: {format_8h_slot(slot)}"
    print_comparison_stats(slot_stats, label, timeframes, tf_labels)

# Print 6H comparison
for slot in range(4):
    slot_stats = {tf: results[tf]['6h'][slot] for tf in timeframes}
    label = f"6H SLOT {slot}: {format_6h_slot(slot)}"
    print_comparison_stats(slot_stats, label, timeframes, tf_labels)

# Print 4H comparison
for slot in range(6):
    slot_stats = {tf: results[tf]['4h'][slot] for tf in timeframes}
    label = f"4H SLOT {slot}: {format_4h_slot(slot)}"
    print_comparison_stats(slot_stats, label, timeframes, tf_labels)

# Print Hourly comparison (all 24 hours)
for hour in range(24):
    hour_stats = {tf: results[tf]['hourly'][hour] for tf in timeframes}
    label = f"TRADING HOUR {hour}: {format_hour(hour)}"
    print_comparison_stats(hour_stats, label, timeframes, tf_labels)

print("\n" + "=" * 100)
print("COMPLETE TIMEFRAME COMPARISON ANALYSIS WITH 60 CANDLE LOOKBACK - COMPLETE")
print("=" * 100)
sys.stdout.flush()

STATISTICAL LEVEL ANALYSIS - 60 LOOKBACK WITH TIMEFRAME COMPARISON

Loading data from: C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv
✓ Loaded 4,974,698 rows

Creating datetime index...
✓ Datetime index created

Filtering to 2013-2025...
✓ Filtered to 4,161,727 rows

Resampling to 5-minute bars...
✓ Resampled to 854,417 5-minute bars

Timeframe cutoffs:
  Full dataset: 2013-01-02 to 2025-03-14
  5-year: 2020-03-15 to 2025-03-14
  1-year: 2024-03-14 to 2025-03-14

Dataset sizes:
  Full: 854,417 bars
  5-year: 353,363 bars
  1-year: 70,648 bars

RUNNING ANALYSIS FOR ALL TIMEFRAMES
Lookback period: 60 candles
Timing analysis: ENABLED

ANALYZING TIMEFRAME: 1-YEAR
Adding trading hour and slot columns for 1-YEAR...
✓ Columns added

PHASE 1: Collecting historical data for 1-YEAR
  Aggregating Hourly candles...
  ✓ Created 5,891 Hourly candles
  Aggregating 4H candles...
  ✓ Created 1,537 4H candles
  Aggregating 6H candles...
  ✓ Created 1,032 6H candles
  Aggregating 8H candles...
  ✓ Cre

  Hourly slot 4: analyzing 3,072 candles... ✓
  Hourly slot 5: analyzing 3,072 candles... ✓
  Hourly slot 6: analyzing 3,072 candles... ✓
  Hourly slot 7: analyzing 3,072 candles... ✓
  Hourly slot 8: analyzing 3,072 candles... ✓
  Hourly slot 9: analyzing 3,072 candles... ✓
  Hourly slot 10: analyzing 3,072 candles... ✓
  Hourly slot 11: analyzing 3,072 candles... ✓
  Hourly slot 12: analyzing 3,075 candles... ✓
  Hourly slot 13: analyzing 3,076 candles... ✓
  Hourly slot 14: analyzing 3,075 candles... ✓
  Hourly slot 15: analyzing 3,076 candles... ✓
  Hourly slot 16: analyzing 3,070 candles... ✓
  Hourly slot 17: analyzing 3,068 candles... ✓
  Hourly slot 18: analyzing 3,068 candles... ✓
  Hourly slot 19: analyzing 3,000 candles... ✓
  Hourly slot 20: analyzing 2,973 candles... ✓
  Hourly slot 21: analyzing 2,973 candles... ✓
  Hourly slot 22: analyzing 2,973 candles... ✓
  Hourly slot 23: analyzing 650 candles... ✓

  Analyzing 4H candles...
  4H slot 0: analyzing 3,072 candles... ✓

Median Retracement (points)                        27.38           23.00           11.50          
Median Wick-to-Wick Ratio                          186.96%         200.00%         200.00%        

RETRACEMENT ANALYSIS - MANIP HIGH - MEAN-BASED (Median):
----------------------------------------------------------------------------------------------------
Median Retracement (points)                        28.75           23.50           12.00          
Median Wick-to-Wick Ratio                          131.71%         151.07%         151.08%        

RETRACEMENT ANALYSIS - MANIP LOW - MEDIAN-BASED (Median):
----------------------------------------------------------------------------------------------------
Median Retracement (points)                        28.75           23.25           11.50          
Median Wick-to-Wick Ratio                          217.86%         200.00%         200.00%        

RETRACEMENT ANALYSIS - MANIP LOW - MEAN-BASED (Median):
------------------------------

Median Retracement (points)                        123.12          99.62           44.00          
Median Wick-to-Wick Ratio                          149.28%         159.06%         151.46%        

RETRACEMENT ANALYSIS - MANIP LOW - MEDIAN-BASED (Median):
----------------------------------------------------------------------------------------------------
Median Retracement (points)                        139.75          104.00          47.50          
Median Wick-to-Wick Ratio                          295.80%         227.07%         218.63%        

RETRACEMENT ANALYSIS - MANIP LOW - MEAN-BASED (Median):
----------------------------------------------------------------------------------------------------
Median Retracement (points)                        144.25          108.50          48.75          
Median Wick-to-Wick Ratio                          212.34%         180.23%         175.78%        

TIMING - MANIP HIGH HITS - MEDIAN-BASED:
----------------------------------------------

In [5]:
# ============================================================
# ATR NORMALISATION ANALYSIS
# ============================================================
# This cell should be run AFTER the main analysis (results dict must exist).
#
# What this does:
#   1. Computes a rolling 14-period ATR on DAILY candles for every bar in
#      the dataset.  The ATR is calculated using the standard Wilder method:
#          ATR[i] = max(high-low, |high-prev_close|, |low-prev_close|)
#      then smoothed with a 14-period Wilder EMA (alpha = 1/14).
#
#   2. For every candle in every slot / timeframe that was already analysed,
#      it looks up the ATR value that was current at the START of that candle
#      (i.e. the ATR as of the previous daily close, so there is no look-ahead).
#
#   3. It then re-expresses the key level distances as ATR multiples:
#          atr_multiple = level_distance_in_points / daily_atr_at_that_time
#
#   4. Outputs a summary table (same 1Y / 5Y / Full structure as the main
#      notebook) showing median ATR multiples for:
#          - Manipulation Wick
#          - Distribution Distance
#          - Max Range
#      across all candle timeframes.
#
#   5. Also computes ATR REGIME buckets (Low / Medium / High volatility) and
#      reports level hit-rates broken down by regime, so you can see whether
#      the manipulation-to-distribution pattern is more reliable in certain
#      conditions.
#
# PRACTICAL OUTPUT:
#   The final 'atr_levels' dict gives you today's adaptive levels.  Just
#   multiply the median ATR multiples by the current 14-day ATR to get
#   live point-distance targets.
# ============================================================

import numpy as np
import pandas as pd
import sys

print("=" * 80)
print("ATR NORMALISATION ANALYSIS")
print("=" * 80)

# ──────────────────────────────────────────────────────────────
# STEP 1: Build a daily ATR series from the 5-min base data
# ──────────────────────────────────────────────────────────────

ATR_PERIOD = 14   # standard Wilder period – change if desired
REGIME_LOW_PCT  = 33   # ATR percentile boundary for Low regime
REGIME_HIGH_PCT = 67   # ATR percentile boundary for High regime

print(f"\nBuilding {ATR_PERIOD}-period Wilder ATR on daily candles...")
sys.stdout.flush()

# Re-aggregate 5-min data to daily candles using the same NY trading day
# definition that the main notebook uses (day starts at 18:00 ET).
daily_agg = df_full.groupby('NY_Trading_Day').agg(
    Open=('Open', 'first'),
    High=('High', 'max'),
    Low=('Low', 'min'),
    Close=('Close', 'last')
).sort_index()

# True Range
daily_agg['prev_close'] = daily_agg['Close'].shift(1)
daily_agg['tr'] = daily_agg[['High', 'Low', 'prev_close']].apply(
    lambda r: max(
        r['High'] - r['Low'],
        abs(r['High'] - r['prev_close']) if not pd.isna(r['prev_close']) else 0,
        abs(r['Low']  - r['prev_close']) if not pd.isna(r['prev_close']) else 0,
    ),
    axis=1
)

# Wilder smoothing (equivalent to EMA with alpha = 1/period)
alpha = 1.0 / ATR_PERIOD
daily_agg['atr'] = daily_agg['tr'].ewm(alpha=alpha, min_periods=ATR_PERIOD, adjust=False).mean()

# Shift by 1 so the ATR is "as of the PREVIOUS close" – no look-ahead
daily_agg['atr_prev'] = daily_agg['atr'].shift(1)

# Build a quick lookup: NY_Trading_Day (Timestamp) → ATR value
atr_lookup = daily_agg['atr_prev'].dropna().to_dict()

print(f"✓ ATR series built: {len(atr_lookup):,} daily values")
print(f"  ATR range: {daily_agg['atr'].min():.1f} – {daily_agg['atr'].max():.1f} points")
print(f"  Median ATR (full): {daily_agg['atr'].median():.1f} points")
sys.stdout.flush()

# ──────────────────────────────────────────────────────────────
# STEP 2: Compute ATR regime boundaries (on full dataset)
# ──────────────────────────────────────────────────────────────

atr_values_full = daily_agg['atr'].dropna().values
regime_low_threshold  = np.percentile(atr_values_full, REGIME_LOW_PCT)
regime_high_threshold = np.percentile(atr_values_full, REGIME_HIGH_PCT)

print(f"\nATR Regime boundaries ({REGIME_LOW_PCT}th / {REGIME_HIGH_PCT}th percentile):")
print(f"  Low  ATR  : < {regime_low_threshold:.1f} points")
print(f"  Medium ATR: {regime_low_threshold:.1f} – {regime_high_threshold:.1f} points")
print(f"  High ATR  : > {regime_high_threshold:.1f} points")
sys.stdout.flush()

def get_regime(atr_val):
    if pd.isna(atr_val) or atr_val <= 0:
        return None
    if atr_val < regime_low_threshold:
        return 'Low'
    elif atr_val <= regime_high_threshold:
        return 'Medium'
    else:
        return 'High'

# ──────────────────────────────────────────────────────────────
# STEP 3: Helper – extract ATR-normalised stats from a
#          historical dict (same format as main notebook)
# ──────────────────────────────────────────────────────────────

def compute_atr_stats(hist, lookback=60):
    """
    Given a historical dict (manipulation_wicks, distribution_distances,
    max_ranges, timestamps, ...) re-express level distances as ATR multiples
    and compute regime-conditioned hit rates.

    'timestamps' contains the candle ID string (e.g. '2024-01-15_2').
    We parse the date portion to look up the ATR.

    Returns a dict with summary statistics.
    """
    n = len(hist['manipulation_wicks'])
    if n <= lookback:
        return None

    manip_atr   = []
    dist_atr    = []
    max_atr     = []
    regimes     = []

    # Per-regime hit counts  {regime: {metric: [hit, total]}}
    regime_hits = {r: {
        'manip_high_median': [0, 0],
        'manip_low_median':  [0, 0],
        'dist_high_median':  [0, 0],
        'dist_low_median':   [0, 0],
    } for r in ['Low', 'Medium', 'High']}

    for i in range(lookback, n):
        # Parse the NY trading day from the timestamp string
        ts_str = str(hist['timestamps'][i])
        # timestamps look like '2024-01-15' or '2024-01-15_2' etc.
        date_part = ts_str.split('_')[0]
        try:
            ny_day = pd.Timestamp(date_part)
        except Exception:
            continue

        atr_val = atr_lookup.get(ny_day, np.nan)
        if pd.isna(atr_val) or atr_val <= 0:
            continue

        # Look-back window for this candle
        lookback_manip = hist['manipulation_wicks'][i-lookback:i]
        lookback_dist  = hist['distribution_distances'][i-lookback:i]
        lookback_max   = hist['max_ranges'][i-lookback:i]

        median_manip = np.median(lookback_manip)
        median_dist  = np.median(lookback_dist)
        median_max   = np.median(lookback_max)

        # ATR multiples (level distance ÷ ATR)
        manip_atr.append(median_manip / atr_val)
        dist_atr.append(median_dist   / atr_val)
        max_atr.append(median_max     / atr_val)

        # Regime label
        regime = get_regime(atr_val)
        regimes.append(regime)

        # Regime-conditioned hit rates (median-based only, for brevity)
        if regime is None:
            continue

        current_open  = hist['opens'][i]
        current_high  = hist['highs'][i]
        current_low   = hist['lows'][i]

        manip_high_med = current_open + median_manip
        manip_low_med  = current_open - median_manip
        dist_high_med  = current_open + median_dist + median_manip
        dist_low_med   = current_open - median_dist - median_manip

        rh = regime_hits[regime]
        rh['manip_high_median'][1] += 1
        rh['manip_low_median'][1]  += 1
        rh['dist_high_median'][1]  += 1
        rh['dist_low_median'][1]   += 1

        if current_high >= manip_high_med:
            rh['manip_high_median'][0] += 1
        if current_low <= manip_low_med:
            rh['manip_low_median'][0]  += 1
        if current_high >= dist_high_med:
            rh['dist_high_median'][0]  += 1
        if current_low <= dist_low_med:
            rh['dist_low_median'][0]   += 1

    if len(manip_atr) == 0:
        return None

    # Regime distribution
    regime_counts = {r: regimes.count(r) for r in ['Low', 'Medium', 'High']}
    total_with_regime = sum(regime_counts.values())

    # Hit rates per regime
    def hit_rate(r, key):
        hits, total = regime_hits[r][key]
        return (hits / total * 100) if total > 0 else np.nan

    return {
        # ATR multiple distributions
        'manip_atr_median':  np.median(manip_atr),
        'manip_atr_mean':    np.mean(manip_atr),
        'manip_atr_p25':     np.percentile(manip_atr, 25),
        'manip_atr_p75':     np.percentile(manip_atr, 75),

        'dist_atr_median':   np.median(dist_atr),
        'dist_atr_mean':     np.mean(dist_atr),
        'dist_atr_p25':      np.percentile(dist_atr, 25),
        'dist_atr_p75':      np.percentile(dist_atr, 75),

        'max_atr_median':    np.median(max_atr),
        'max_atr_p25':       np.percentile(max_atr, 25),
        'max_atr_p75':       np.percentile(max_atr, 75),

        'total_dist_atr_median': np.median([m + d for m, d in zip(manip_atr, dist_atr)]),

        # Regime distribution %
        'regime_pct_low':    regime_counts['Low']    / total_with_regime * 100 if total_with_regime else np.nan,
        'regime_pct_medium': regime_counts['Medium'] / total_with_regime * 100 if total_with_regime else np.nan,
        'regime_pct_high':   regime_counts['High']   / total_with_regime * 100 if total_with_regime else np.nan,

        # Hit rates by regime
        'low_manip_high_hit':   hit_rate('Low',    'manip_high_median'),
        'low_manip_low_hit':    hit_rate('Low',    'manip_low_median'),
        'low_dist_high_hit':    hit_rate('Low',    'dist_high_median'),
        'low_dist_low_hit':     hit_rate('Low',    'dist_low_median'),

        'med_manip_high_hit':   hit_rate('Medium', 'manip_high_median'),
        'med_manip_low_hit':    hit_rate('Medium', 'manip_low_median'),
        'med_dist_high_hit':    hit_rate('Medium', 'dist_high_median'),
        'med_dist_low_hit':     hit_rate('Medium', 'dist_low_median'),

        'high_manip_high_hit':  hit_rate('High',   'manip_high_median'),
        'high_manip_low_hit':   hit_rate('High',   'manip_low_median'),
        'high_dist_high_hit':   hit_rate('High',   'dist_high_median'),
        'high_dist_low_hit':    hit_rate('High',   'dist_low_median'),
    }

# ──────────────────────────────────────────────────────────────
# STEP 4: Run over all timeframes and all three lookback windows
# ──────────────────────────────────────────────────────────────

# Map the results keys back to historical dicts
# (re-run historical aggregation using the already-computed results structure)
# We re-use the historical dicts that were built inside analyze_timeframe.
# Since those are local to that function we need to re-build them here.
# To avoid duplicating all that code we call analyze_timeframe with timing
# disabled and extract the historical data we need.
#
# FASTER APPROACH: we already have the stats in `results`.  We can reconstruct
# ATR-normalised stats directly from the original dataframes using the same
# aggregation logic, but this time passing through atr_lookup.

print("\nComputing ATR-normalised stats for all timeframes...")
print("(This re-aggregates candles – takes ~1-2 minutes)")
sys.stdout.flush()

LOOKBACK = 60  # must match the lookback used in the main analysis

def build_historical_for_atr(df_data, slot_type):
    """
    Lightweight re-aggregation that returns a dict of historical dicts
    suitable for compute_atr_stats.  slot_type is one of:
    'daily', 'hourly', '4h', '6h', '8h', '12h'
    """
    # Ensure slot columns exist
    df_data = df_data.copy()
    df_data['trading_hour']     = df_data.index.map(get_trading_hour)
    df_data['trading_4h_slot']  = df_data.index.map(get_4h_slot)
    df_data['trading_6h_slot']  = df_data.index.map(get_6h_slot)
    df_data['trading_8h_slot']  = df_data.index.map(get_8h_slot)
    df_data['trading_12h_slot'] = df_data.index.map(get_12h_slot)
    df_data['hour_id']  = df_data['NY_Trading_Day'].astype(str) + '_' + df_data['trading_hour'].astype(str)
    df_data['4h_id']    = df_data['NY_Trading_Day'].astype(str) + '_' + df_data['trading_4h_slot'].astype(str)
    df_data['6h_id']    = df_data['NY_Trading_Day'].astype(str) + '_' + df_data['trading_6h_slot'].astype(str)
    df_data['8h_id']    = df_data['NY_Trading_Day'].astype(str) + '_' + df_data['trading_8h_slot'].astype(str)
    df_data['12h_id']   = df_data['NY_Trading_Day'].astype(str) + '_' + df_data['trading_12h_slot'].astype(str)
    df_data['daily_id'] = df_data['NY_Trading_Day'].astype(str)

    cfg = {
        'daily':  ('daily_id',  None,                None),
        'hourly': ('hour_id',   'trading_hour',       24),
        '4h':     ('4h_id',     'trading_4h_slot',    6),
        '6h':     ('6h_id',     'trading_6h_slot',    4),
        '8h':     ('8h_id',     'trading_8h_slot',    3),
        '12h':    ('12h_id',    'trading_12h_slot',   2),
    }
    id_col, slot_col, num_slots = cfg[slot_type]

    if num_slots is None:
        # Daily
        grp = df_data.groupby(id_col, sort=False).agg(
            Open=('Open','first'), High=('High','max'),
            Low=('Low','min'),  Close=('Close','last')
        ).reset_index()
        grp['manip_wick']    = grp.apply(lambda r: calculate_manipulation_wick(r['Open'], r['High'], r['Low'], r['Close']), axis=1)
        grp['dist_distance'] = grp.apply(lambda r: calculate_distribution_distance(r['Open'], r['High'], r['Low'], r['Close']), axis=1)
        grp['max_range']     = grp.apply(lambda r: calculate_max_range_from_open(r['Open'], r['High'], r['Low']), axis=1)
        grp['high_range']    = grp.apply(lambda r: calculate_high_range_from_open(r['Open'], r['High']), axis=1)
        grp['low_range']     = grp.apply(lambda r: calculate_low_range_from_open(r['Open'], r['Low']), axis=1)
        hist = {
            None: {
                'manipulation_wicks':    grp['manip_wick'].tolist(),
                'distribution_distances': grp['dist_distance'].tolist(),
                'max_ranges':            grp['max_range'].tolist(),
                'high_ranges':           grp['high_range'].tolist(),
                'low_ranges':            grp['low_range'].tolist(),
                'opens':      grp['Open'].tolist(),
                'highs':      grp['High'].tolist(),
                'lows':       grp['Low'].tolist(),
                'closes':     grp['Close'].tolist(),
                'timestamps': grp[id_col].tolist(),
                'candle_ranges': (grp['High'] - grp['Low']).tolist(),
            }
        }
    else:
        grp = df_data.groupby([id_col, slot_col], sort=False).agg(
            Open=('Open','first'), High=('High','max'),
            Low=('Low','min'),  Close=('Close','last')
        ).reset_index()
        hist = {}
        for slot in range(num_slots):
            sc = grp[grp[slot_col] == slot].copy()
            if len(sc) == 0:
                hist[slot] = {k: [] for k in ['manipulation_wicks','distribution_distances',
                                               'max_ranges','high_ranges','low_ranges',
                                               'opens','highs','lows','closes','timestamps','candle_ranges']}
                continue
            sc['manip_wick']    = sc.apply(lambda r: calculate_manipulation_wick(r['Open'], r['High'], r['Low'], r['Close']), axis=1)
            sc['dist_distance'] = sc.apply(lambda r: calculate_distribution_distance(r['Open'], r['High'], r['Low'], r['Close']), axis=1)
            sc['max_range']     = sc.apply(lambda r: calculate_max_range_from_open(r['Open'], r['High'], r['Low']), axis=1)
            sc['high_range']    = sc.apply(lambda r: calculate_high_range_from_open(r['Open'], r['High']), axis=1)
            sc['low_range']     = sc.apply(lambda r: calculate_low_range_from_open(r['Open'], r['Low']), axis=1)
            hist[slot] = {
                'manipulation_wicks':    sc['manip_wick'].tolist(),
                'distribution_distances': sc['dist_distance'].tolist(),
                'max_ranges':            sc['max_range'].tolist(),
                'high_ranges':           sc['high_range'].tolist(),
                'low_ranges':            sc['low_range'].tolist(),
                'opens':      sc['Open'].tolist(),
                'highs':      sc['High'].tolist(),
                'lows':       sc['Low'].tolist(),
                'closes':     sc['Close'].tolist(),
                'timestamps': sc[id_col].tolist(),
                'candle_ranges': (sc['High'] - sc['Low']).tolist(),
            }
    return hist

# Run for all three timeframes and all candle types
atr_results = {}

tf_map = {'1y': df_1y, '5y': df_5y, 'full': df_full}
slot_types = ['daily', '12h', '8h', '6h', '4h', 'hourly']

for tf_key, df_tf in tf_map.items():
    atr_results[tf_key] = {}
    for st in slot_types:
        print(f"  {tf_key.upper()} / {st}...", end=' ')
        sys.stdout.flush()
        hist_dict = build_historical_for_atr(df_tf, st)
        atr_results[tf_key][st] = {}
        for slot_key, hist in hist_dict.items():
            r = compute_atr_stats(hist, lookback=LOOKBACK)
            atr_results[tf_key][st][slot_key] = r
        print("✓")
        sys.stdout.flush()

print("\n✓ ATR normalisation complete")

# ──────────────────────────────────────────────────────────────
# STEP 5: Print summary tables
# ──────────────────────────────────────────────────────────────

def fmt(val, decimals=3):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return "N/A"
    return f"{val:.{decimals}f}x"

def fmt_pct(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return "N/A"
    return f"{val:.1f}%"

W = 100
SEP = "=" * W
sep = "-" * W

def print_atr_table(slot_type, slot_key, slot_label):
    print(f"\n{SEP}")
    print(f"ATR-NORMALISED LEVELS — {slot_label}")
    print(SEP)
    print(f"{'Metric':<52} {'1-Year':<16} {'5-Year':<16} {'Full':<16}")
    print(sep)

    rows = [
        ("ATR MULTIPLES (median of 60-candle rolling ATR multiple):", None),
        ("Manipulation Wick  – Median ATR multiple",   lambda r: fmt(r['manip_atr_median'])),
        ("Manipulation Wick  – Mean ATR multiple",     lambda r: fmt(r['manip_atr_mean'])),
        ("Manipulation Wick  – 25th pctile",           lambda r: fmt(r['manip_atr_p25'])),
        ("Manipulation Wick  – 75th pctile",           lambda r: fmt(r['manip_atr_p75'])),
        ("Distribution Dist  – Median ATR multiple",   lambda r: fmt(r['dist_atr_median'])),
        ("Distribution Dist  – Mean ATR multiple",     lambda r: fmt(r['dist_atr_mean'])),
        ("Distribution Dist  – 25th pctile",           lambda r: fmt(r['dist_atr_p25'])),
        ("Distribution Dist  – 75th pctile",           lambda r: fmt(r['dist_atr_p75'])),
        ("Max Range          – Median ATR multiple",   lambda r: fmt(r['max_atr_median'])),
        ("Max Range          – 25th pctile",           lambda r: fmt(r['max_atr_p25'])),
        ("Max Range          – 75th pctile",           lambda r: fmt(r['max_atr_p75'])),
        ("Total (Manip+Dist) – Median ATR multiple",   lambda r: fmt(r['total_dist_atr_median'])),
        ("", None),
        ("VOLATILITY REGIME DISTRIBUTION:", None),
        ("Low  ATR regime  (bottom 33%)",  lambda r: fmt_pct(r['regime_pct_low'])),
        ("Medium ATR regime (middle 34%)", lambda r: fmt_pct(r['regime_pct_medium'])),
        ("High ATR regime  (top 33%)",     lambda r: fmt_pct(r['regime_pct_high'])),
        ("", None),
        ("MANIPULATION HIT RATES BY REGIME (median-based levels):", None),
        ("Low  Vol — Manip High Hit",  lambda r: fmt_pct(r['low_manip_high_hit'])),
        ("Low  Vol — Manip Low Hit",   lambda r: fmt_pct(r['low_manip_low_hit'])),
        ("Low  Vol — Dist  High Hit",  lambda r: fmt_pct(r['low_dist_high_hit'])),
        ("Low  Vol — Dist  Low Hit",   lambda r: fmt_pct(r['low_dist_low_hit'])),
        ("Med  Vol — Manip High Hit",  lambda r: fmt_pct(r['med_manip_high_hit'])),
        ("Med  Vol — Manip Low Hit",   lambda r: fmt_pct(r['med_manip_low_hit'])),
        ("Med  Vol — Dist  High Hit",  lambda r: fmt_pct(r['med_dist_high_hit'])),
        ("Med  Vol — Dist  Low Hit",   lambda r: fmt_pct(r['med_dist_low_hit'])),
        ("High Vol — Manip High Hit",  lambda r: fmt_pct(r['high_manip_high_hit'])),
        ("High Vol — Manip Low Hit",   lambda r: fmt_pct(r['high_manip_low_hit'])),
        ("High Vol — Dist  High Hit",  lambda r: fmt_pct(r['high_dist_high_hit'])),
        ("High Vol — Dist  Low Hit",   lambda r: fmt_pct(r['high_dist_low_hit'])),
    ]

    for label, fn in rows:
        if fn is None:
            # Section header
            if label:
                print(f"\n{label}")
                print(sep)
            else:
                print()
        else:
            vals = []
            for tf in ['1y', '5y', 'full']:
                r = atr_results[tf][slot_type].get(slot_key)
                vals.append(fn(r) if r else "N/A")
            print(f"  {label:<50} {vals[0]:<16} {vals[1]:<16} {vals[2]:<16}")

# Daily
print_atr_table('daily', None, "DAILY CANDLES")

# 12H slots
slot_labels_12h = {0: "12H SLOT 0: 18:00–06:00", 1: "12H SLOT 1: 06:00–18:00"}
for s, lbl in slot_labels_12h.items():
    print_atr_table('12h', s, lbl)

# 8H slots
slot_labels_8h = {0: "8H SLOT 0: 18:00–02:00", 1: "8H SLOT 1: 02:00–10:00", 2: "8H SLOT 2: 10:00–18:00"}
for s, lbl in slot_labels_8h.items():
    print_atr_table('8h', s, lbl)

# 6H slots
slot_labels_6h = {0: "6H SLOT 0: 18:00–00:00", 1: "6H SLOT 1: 00:00–06:00",
                  2: "6H SLOT 2: 06:00–12:00", 3: "6H SLOT 3: 12:00–18:00"}
for s, lbl in slot_labels_6h.items():
    print_atr_table('6h', s, lbl)

# 4H slots
slot_labels_4h = {0: "4H SLOT 0: 18:00–22:00", 1: "4H SLOT 1: 22:00–02:00",
                  2: "4H SLOT 2: 02:00–06:00", 3: "4H SLOT 3: 06:00–10:00",
                  4: "4H SLOT 4: 10:00–14:00", 5: "4H SLOT 5: 14:00–18:00"}
for s, lbl in slot_labels_4h.items():
    print_atr_table('4h', s, lbl)

# ──────────────────────────────────────────────────────────────
# STEP 6: PRACTICAL OUTPUT – today's adaptive levels
# ──────────────────────────────────────────────────────────────

print(f"\n{'=' * 80}")
print("ADAPTIVE LEVEL CALCULATOR")
print("Multiply these ATR multiples by today's 14-day ATR to get point targets.")
print(f"{'=' * 80}")

# Use most recent ATR value
latest_atr = daily_agg['atr'].dropna().iloc[-1]
latest_date = daily_agg['atr'].dropna().index[-1]

print(f"\nLatest 14-day ATR: {latest_atr:.2f} points  (as of {latest_date.date()})")
print()

# Print daily adaptive levels using 1Y multiples (most recent regime)
r_daily_1y = atr_results['1y']['daily'].get(None)
if r_daily_1y:
    print(f"  Daily candle targets (based on 1-year ATR multiples × {latest_atr:.0f} ATR):")
    print(f"    Manipulation wick (median): ±{r_daily_1y['manip_atr_median'] * latest_atr:.1f} pts")
    print(f"    Distribution level (median): ±{r_daily_1y['total_dist_atr_median'] * latest_atr:.1f} pts from open")
    print(f"    Max range (median):          ±{r_daily_1y['max_atr_median'] * latest_atr:.1f} pts from open")

print(f"\n{'=' * 80}")
print("✓ ATR NORMALISATION COMPLETE")
print("  Results stored in: atr_results[tf][slot_type][slot_key]")
print("  ATR series stored in: daily_agg['atr']")
print("  Regime thresholds: regime_low_threshold, regime_high_threshold")
print(f"{'=' * 80}")

ATR NORMALISATION ANALYSIS

Building 14-period Wilder ATR on daily candles...
✓ ATR series built: 3,769 daily values
  ATR range: 24.8 – 529.9 points
  Median ATR (full): 109.9 points

ATR Regime boundaries (33th / 67th percentile):
  Low  ATR  : < 60.8 points
  Medium ATR: 60.8 – 202.0 points
  High ATR  : > 202.0 points

Computing ATR-normalised stats for all timeframes...
(This re-aggregates candles – takes ~1-2 minutes)
  1Y / daily... ✓
  1Y / 12h... ✓
  1Y / 8h... ✓
  1Y / 6h... ✓
  1Y / 4h... ✓
  1Y / hourly... ✓
  5Y / daily... ✓
  5Y / 12h... ✓
  5Y / 8h... ✓
  5Y / 6h... ✓
  5Y / 4h... ✓
  5Y / hourly... ✓
  FULL / daily... ✓
  FULL / 12h... ✓
  FULL / 8h... ✓
  FULL / 6h... ✓
  FULL / 4h... ✓
  FULL / hourly... ✓

✓ ATR normalisation complete

ATR-NORMALISED LEVELS — DAILY CANDLES
Metric                                               1-Year           5-Year           Full            
---------------------------------------------------------------------------------------------